In [20]:
# -*- coding: utf-8 -*-
"""
================================================================================
SCRIPT 1 — CARE ANOMALY DETECTION (FULL CLEAN REBUILD, IMPROVED)
================================================================================
Purpose
-------
Stage 1 anomaly detection for the CARE benchmark using:
  - exact sensor names from feature_description.csv
  - farm-specific global Isolation Forest normal behaviour models
  - optional subsystem models (Farm B only)
  - full CARE score computation
  - dissertation-quality outputs

Key design choices
------------------
1. Sensor renaming uses exact descriptions from the CARE feature description file.
2. Training uses only NORMAL data (status 0/2 in training windows).
3. Prediction windows are not status-filtered before scoring.
4. Farm A is evaluated without status filtering in criticality logic
   (consistent with CARE / Zenodo note).
5. Farm B subsystem alarms are restricted to transformer only by default,
   because broad subsystem fusion was inflating false alarms.
6. Global and subsystem alarms are separated cleanly.
7. Exact sensor names are preserved in registry and downstream plots.

Author: Peter
Dataset root: E:\\project\\CARE_To_Compare
"""

# ============================================================================
# IMPORTS
# ============================================================================

import os
import gc
import re
import json
import warnings
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")


# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_PATH = r"E:\project\CARE_To_Compare"

FARMS = {
    "A": os.path.join(BASE_PATH, "Wind Farm A"),
    "B": os.path.join(BASE_PATH, "Wind Farm B"),
    "C": os.path.join(BASE_PATH, "Wind Farm C"),
}

EVENT_INFO_PATH = os.path.join(BASE_PATH, "df_all__1_.csv")

OUTPUT_DIR = os.path.join(BASE_PATH, "stage1_clean_outputs")
PLOT_DIR   = os.path.join(OUTPUT_DIR, "plots")
REPORT_DIR = os.path.join(OUTPUT_DIR, "reports")

for d in [OUTPUT_DIR, PLOT_DIR, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)

FARM_NAME_MAP = {"Farm A": "A", "Farm B": "B", "Farm C": "C"}

NORMAL_STATUS = {0, 2}
META_COLS = {"time_stamp", "asset_id", "id", "train_test", "status_type_id"}

# CARE-oriented parameters
BETA = 0.5
CRITICALITY_THRESHOLD_GLOBAL = 36     # 6 hours
CRITICALITY_THRESHOLD_SUB    = 60     # 10 hours, stricter for subsystem alarms
SMOOTHING_WINDOW             = 6      # 1 hour rolling median

# Isolation Forest / PCA
IF_CONTAMINATION     = 0.01
IF_N_ESTIMATORS      = 200
PCA_VARIANCE_TARGET  = 0.95
MAX_PCA_COMPONENTS   = 150
MIN_PCA_COMPONENTS   = 5

# Farm-specific score thresholds from training score percentiles
FARM_THRESHOLD_PCT = {
    "A": 95,
    "B": 97,
    "C": 99,
}

# Subsystem models
SUBSYSTEM_MODEL_FARMS = {"B"}
MIN_SENSORS_FOR_SUBSYSTEM_MODEL = 4

# Important fix:
# only allow subsystem alarms from these groups to raise event-level alarms
# This prevents Farm B from getting many false alarms from broad groups.
SUBSYSTEM_ALARM_GROUPS = {
    "A": set(),
    "B": {"transformer"},
    "C": set(),
}

# Groups allowed to be trained for analysis / score storage
SUBSYSTEM_TRAIN_GROUPS = {
    "A": set(),
    "B": {"transformer", "gearbox_bearing", "pitch", "electrical", "wind"},
    "C": set(),
}

# Use OR fusion only for score-time anomaly flags inside allowed alarm groups
SCORE_FUSION = "max"

# For feature engineering: keep only top avg columns by variance per farm
TOP_AVG_COLS_PER_FARM = {
    "A": 40,
    "B": 50,
    "C": 60,
}


# ============================================================================
# SUBSYSTEM RULES
# ============================================================================

SUBSYSTEM_RULES = [
    (10,  "gearbox_bearing",  ["planetary bearing", "gearbox bearing",
                               "bearing on high speed shaft"]),
    (11,  "gearbox_oil",      ["gearbox oil", "gear oil", "oil inlet temperature",
                               "oil pressure", "oil level", "oil tank",
                               "oil temperature"]),
    (12,  "gearbox_other",    ["gearbox rotational", "gearbox speed",
                               "gearbox main shaft", "gearbox"]),
    (20,  "generator_bearing",["generator bearing"]),
    (21,  "generator_stator", ["stator winding", "stator"]),
    (22,  "generator_cooling",["generator cooling", "cooling air",
                               "cooling water temp. generator",
                               "generator temperature", "cooling air inlet"]),
    (23,  "generator_other",  ["generator acceleration", "generator angle",
                               "generator rpm", "generator converter",
                               "generator rms", "generator"]),
    (30,  "rotor_bearing",    ["rotor bearing", "axial bearing",
                               "rotor bearing inner ring"]),
    (40,  "hydraulic",        ["hydraulic pressure", "hydraulic oil",
                               "hydraulic pump", "hydraulic tank",
                               "hydraulic group", "aggregate system pressure",
                               "azimuth circuit", "azimuth memory",
                               "rotor brake", "locking memory",
                               "pollution indicator", "oil container"]),
    (50,  "pitch",            ["pitch angle", "pitch motor", "pitch axis",
                               "position motor axis", "position rotor blade",
                               "motor temperature kty", "motor temperature pt",
                               "axis 1 motor", "axis 2 motor", "axis 3 motor",
                               "axis 1 voltage", "axis 2 voltage", "axis 3 voltage",
                               "battery charge", "battery discharge",
                               "rpm motor axis", "min pitch",
                               "motor current axis", "motor temperature axis"]),
    (60,  "vibration",        ["vibration", "drive train vibration",
                               "tower vibration"]),
    (70,  "transformer",      ["transformer", "transformator",
                               "hv transformer", "transformer cell"]),
    (80,  "electrical",       ["voltage", "current", "frequency",
                               "active power", "reactive power",
                               "apparent power", "grid power", "grid voltage",
                               "grid frequency", "available power",
                               "inverter torque", "cable load",
                               "dc link voltage", "internal consumption"]),
    (90,  "wind",             ["wind speed", "wind direction", "anemometer",
                               "estimated windspeed", "absolute wind",
                               "relative wind"]),
    (100, "environment",      ["ambient temperature", "outside temperature",
                               "nacelle temperature", "nacelle external",
                               "hub temperature", "board temperature",
                               "electrical cabinet", "converter room",
                               "control box temperature", "nacelle outside",
                               "platform temperature"]),
    (110, "cooling_water",    ["water temperature", "water pressure",
                               "water flow", "water conductivity",
                               "water cooler", "cooling water",
                               "flow meter stator"]),
    (120, "control",          ["yaw", "nacelle direction",
                               "estimated tower", "compensation angle",
                               "rotor speed", "rotor rpm", "bypass valve",
                               "aeration", "pressure difference",
                               "rotational speed aerator", "cable load"]),
    (130, "temperature",      ["temperature", "temp."]),
]


# ============================================================================
# SENSOR RENAMING / REGISTRY
# ============================================================================

STAT_MAP = {
    "average": "avg",
    "std_dev": "std",
    "maximum": "max",
    "minimum": "min",
}


def _clean_desc(text):
    if pd.isna(text) or str(text).strip() in ("", "nan"):
        return "unknown"
    text = str(text)
    replacements = [
        ("���C", "degC"), ("°C", "degC"), ("ºC", "degC"), ("½C", "degC"),
        ("Celsius", "degC"), ("���", "deg"), ("°", "deg"), ("º", "deg"),
        ("½", "")
    ]
    for old, new in replacements:
        text = text.replace(old, new)
    text = re.sub(r"[^\x00-\x7F]+", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", "_", text.strip())
    return re.sub(r"_+", "_", text).strip("_")


def _clean_unit(unit):
    if pd.isna(unit) or str(unit).strip().lower() in ("", "nan", "-", "none"):
        return ""
    unit = str(unit)
    replacements = [
        ("���C", "degC"), ("°C", "degC"), ("ºC", "degC"), ("½C", "degC"),
        ("Celsius", "degC"), ("°", "deg"), ("º", "deg"), ("½", ""),
        ("/", "_per_"), ("%", "pct"), (" ", ""), ("^", ""), (".", "")
    ]
    for old, new in replacements:
        unit = unit.replace(old, new)
    unit = re.sub(r"[^\x00-\x7F]+", "", unit)
    return re.sub(r"[^\w]", "", unit)


def _assign_group(description, is_counter=False, is_angle=False):
    if is_counter:
        return "counter"
    if is_angle:
        return "angle"
    desc_lower = description.lower()
    for _, group, keywords in sorted(SUBSYSTEM_RULES, key=lambda x: x[0]):
        if any(kw in desc_lower for kw in keywords):
            return group
    return "other"


def build_sensor_registry(farm_id):
    desc_path = os.path.join(FARMS[farm_id], "feature_description.csv")
    df = None
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(desc_path, sep=";", encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    if df is None:
        raise RuntimeError(f"Could not read feature_description.csv for Farm {farm_id}")

    rename_dict = {}
    group_dict = {}
    rows = []
    seen_new_names = {}

    for _, row in df.iterrows():
        sid        = str(row["sensor_name"]).strip()
        desc_raw   = str(row.get("description", "")).strip()
        unit_raw   = str(row.get("unit", "")).strip()
        is_counter = bool(row.get("is_counter", False))
        is_angle   = bool(row.get("is_angle", False))
        group      = _assign_group(desc_raw, is_counter=is_counter, is_angle=is_angle)

        desc_clean = _clean_desc(desc_raw)
        unit_clean = _clean_unit(unit_raw)

        stats_raw = str(row.get("statistics_type", "")).strip()
        if stats_raw.lower() in ("", "nan"):
            stats = []
        else:
            stats = [s.strip() for s in stats_raw.split(",") if s.strip()]

        entries = []
        if stats:
            for sr in stats:
                stat_clean = STAT_MAP.get(sr, sr)
                entries.append((f"{sid}_{stat_clean}", stat_clean))
        else:
            entries.append((sid, ""))

        for old_col, stat_clean in entries:
            parts = [desc_clean]
            if unit_clean:
                parts.append(unit_clean)
            if stat_clean:
                parts.append(stat_clean)
            new_col = "_".join(parts)

            if new_col in seen_new_names and seen_new_names[new_col] != old_col:
                new_col = f"{new_col}_{sid}"

            seen_new_names[new_col] = old_col
            rename_dict[old_col] = new_col
            group_dict[new_col]  = group

            rows.append({
                "farm": farm_id,
                "sensor_id": sid,
                "old_col": old_col,
                "new_col": new_col,
                "description": desc_raw,
                "unit": unit_raw,
                "stat": stat_clean,
                "group": group,
                "is_counter": is_counter,
                "is_angle": is_angle,
                "use_in_nbm": not (is_counter or is_angle),
            })

    registry_df = pd.DataFrame(rows)
    return rename_dict, group_dict, registry_df


# ============================================================================
# LOAD REGISTRIES
# ============================================================================

print("=" * 80)
print("SCRIPT 1 — CARE ANOMALY DETECTION (FULL CLEAN REBUILD, IMPROVED)")
print("=" * 80)
print(f"Base directory: {BASE_PATH}\n")

registries   = {}
rename_dicts = {}
group_dicts  = {}

for farm in ["A", "B", "C"]:
    rd, gd, reg = build_sensor_registry(farm)
    rename_dicts[farm] = rd
    group_dicts[farm]  = gd
    registries[farm]   = reg

    print(f"Farm {farm} registry:")
    print(f"  Column mappings: {len(rd)}")

    reg_show = reg.copy()
    summary = (
        reg_show
        .groupby("group")
        .agg(
            sensors=("sensor_id", "nunique"),
            cols=("new_col", "count"),
            nbm=("use_in_nbm", "sum")
        )
        .sort_values(["sensors", "cols"], ascending=False)
    )

    for group, row in summary.head(12).iterrows():
        print(f"  {group:22} sensors={int(row['sensors']):3d}  cols={int(row['cols']):3d}  nbm={int(row['nbm']):3d}")
    print()

sensor_registry_path = os.path.join(REPORT_DIR, "sensor_registry_exact.csv")
pd.concat(registries.values(), ignore_index=True).to_csv(sensor_registry_path, index=False)
print(f"Saved sensor registry: {sensor_registry_path}\n")


# ============================================================================
# DATA LOADING
# ============================================================================

def _get_excluded_cols(farm_id):
    reg = registries[farm_id]
    return set(reg.loc[~reg["use_in_nbm"], "new_col"].tolist())


def _handle_zero_fills(df, numeric_cols, excluded_cols):
    """
    Replace long runs of zeros (>=6) with NaN for non-counter, non-angle sensors.
    """
    for col in numeric_cols:
        if col in excluded_cols:
            continue
        vals = df[col].values.copy()
        try:
            vals = vals.astype(np.float32)
        except Exception:
            continue

        n = len(vals)
        i = 0
        while i < n:
            if vals[i] == 0.0:
                start = i
                while i < n and vals[i] == 0.0:
                    i += 1
                if i - start >= 6:
                    vals[start:i] = np.nan
            else:
                i += 1
        df[col] = vals
    return df


def load_event(filepath, farm_id, chunk_size=10000):
    """
    Returns:
        train_df : training rows only, filtered to normal status
        pred_df  : prediction rows only, not status-filtered
    """
    rdict    = rename_dicts[farm_id]
    excluded = _get_excluded_cols(farm_id)
    chunks = []

    for chunk in pd.read_csv(filepath, sep=";", chunksize=chunk_size, low_memory=False, encoding="latin-1"):
        chunk = chunk.rename(columns=rdict)

        num_cols = [c for c in chunk.columns if c not in META_COLS and pd.api.types.is_numeric_dtype(chunk[c])]
        chunk = _handle_zero_fills(chunk, num_cols, excluded)

        if num_cols:
            chunk[num_cols] = chunk[num_cols].astype(np.float32)

        chunks.append(chunk)

    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()

    train_df = df[df["train_test"] == "train"].copy()
    pred_df  = df[df["train_test"] == "prediction"].copy()

    if "status_type_id" in train_df.columns:
        train_df = train_df[train_df["status_type_id"].isin(NORMAL_STATUS)]

    # remove obvious suspicious zero-power normal rows from training
    power_cols = [c for c in train_df.columns
                  if "power" in c.lower() and "avg" in c.lower() and c not in excluded]
    wind_cols  = [c for c in train_df.columns
                  if "wind" in c.lower() and "speed" in c.lower() and "avg" in c.lower()]

    if power_cols and wind_cols:
        p = power_cols[0]
        w = wind_cols[0]
        bad = ((train_df[w] > 3.0) & (train_df[w] < 25.0) & (train_df[p].abs() < 0.01))
        train_df = train_df.loc[~bad].copy()

    del df
    gc.collect()
    return train_df, pred_df


# ============================================================================
# EVENT METADATA
# ============================================================================

event_info = pd.read_csv(EVENT_INFO_PATH)
event_info["farm"] = event_info["Farm"].map(FARM_NAME_MAP)
event_info["event_id"] = event_info["event_id"].astype(str)
event_info["event_label"] = event_info["event_label"].astype(str).str.strip()
event_info["event_description"] = event_info["event_description"].fillna("no description").astype(str)
event_info["event_start"] = pd.to_datetime(event_info["event_start"], errors="coerce")
event_info["event_end"]   = pd.to_datetime(event_info["event_end"],   errors="coerce")

dataset = []
for _, row in event_info.iterrows():
    farm = row["farm"]
    if pd.isna(farm):
        continue
    eid = row["event_id"]
    filepath = os.path.join(FARMS[farm], "datasets", f"{eid}.csv")
    if not os.path.exists(filepath):
        continue

    dataset.append({
        "event_id": eid,
        "farm": farm,
        "label": row["event_label"],
        "description": row["event_description"],
        "filepath": filepath,
        "event_start": row["event_start"],
        "event_end": row["event_end"],
    })

print("Event counts:")
for farm in ["A", "B", "C"]:
    farm_events = [d for d in dataset if d["farm"] == farm]
    n_total = len(farm_events)
    n_anom  = sum(1 for d in farm_events if d["label"] == "anomaly")
    n_norm  = sum(1 for d in farm_events if d["label"] == "normal")
    print(f"  Farm {farm}: total={n_total}  anomaly={n_anom}  normal={n_norm}")
print()


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================

def engineer_features(df, farm_id, avg_cols_ref=None):
    """
    Add rolling mean/std and diff features based on top avg columns.
    """
    excluded = _get_excluded_cols(farm_id)

    avg_cols = [
        c for c in df.columns
        if c not in META_COLS
        and c not in excluded
        and "avg" in c.lower()
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    if avg_cols_ref is None:
        keep_n = TOP_AVG_COLS_PER_FARM.get(farm_id, 40)
        if len(avg_cols) > keep_n:
            avg_cols = df[avg_cols].var().sort_values(ascending=False).head(keep_n).index.tolist()
    else:
        avg_cols = avg_cols_ref

    new_feats = {}
    for col in avg_cols:
        if col in df.columns:
            s = df[col].astype(np.float32)
        else:
            s = pd.Series(np.zeros(len(df), dtype=np.float32), index=df.index)

        new_feats[f"{col}_roll6_mean"] = s.rolling(6, min_periods=1).mean().astype(np.float32)
        new_feats[f"{col}_roll6_std"]  = s.rolling(6, min_periods=1).std().fillna(0).astype(np.float32)
        new_feats[f"{col}_diff1"]      = s.diff(1).fillna(0).astype(np.float32)

    out = pd.concat([df, pd.DataFrame(new_feats, index=df.index)], axis=1)
    return out, avg_cols


def get_feature_columns(df, farm_id):
    excluded = _get_excluded_cols(farm_id)
    feat_cols = [
        c for c in df.columns
        if c not in META_COLS
        and c not in excluded
        and pd.api.types.is_numeric_dtype(df[c])
    ]
    return feat_cols


# ============================================================================
# MODEL TRAINING / SCORING
# ============================================================================

def train_nbm(X_train, threshold_pct):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_train)

    n_comp = min(MAX_PCA_COMPONENTS, X_train.shape[1], max(2, len(X_train) - 1))
    pca = PCA(n_components=n_comp, random_state=42)
    Xp = pca.fit_transform(Xs)

    cumvar = np.cumsum(pca.explained_variance_ratio_)
    n_keep = int(np.searchsorted(cumvar, PCA_VARIANCE_TARGET) + 1)
    n_keep = max(MIN_PCA_COMPONENTS, n_keep)
    n_keep = min(n_keep, n_comp)

    Xp_keep = Xp[:, :n_keep]

    model = IsolationForest(
        n_estimators=IF_N_ESTIMATORS,
        contamination=IF_CONTAMINATION,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(Xp_keep)

    train_scores = -model.score_samples(Xp_keep)
    threshold = float(np.percentile(train_scores, threshold_pct))
    explained = float(cumvar[n_keep - 1])

    del Xs, Xp, Xp_keep, train_scores
    gc.collect()

    return scaler, pca, model, n_keep, threshold, explained


def score_window(pred_df, feat_cols_ref, scaler, pca, model, n_keep, threshold):
    feat_cols = [c for c in feat_cols_ref if c in pred_df.columns]
    X = np.nan_to_num(pred_df[feat_cols].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

    if X.shape[1] < len(feat_cols_ref):
        pad = np.zeros((len(X), len(feat_cols_ref) - X.shape[1]), dtype=np.float32)
        X = np.hstack([X, pad])

    Xs = scaler.transform(X)
    Xp = pca.transform(Xs)[:, :n_keep]
    raw_scores = -model.score_samples(Xp).astype(np.float32)

    smooth_scores = pd.Series(raw_scores).rolling(SMOOTHING_WINDOW, min_periods=1).median().values.astype(np.float32)
    preds = (smooth_scores >= threshold).astype(int)

    del X, Xs, Xp
    gc.collect()
    return preds, smooth_scores


def criticality_algorithm(predictions, status_ids, farm_id):
    crit = 0
    max_crit = 0

    for i in range(len(predictions)):
        status_ok = True if farm_id == "A" else int(status_ids[i]) in NORMAL_STATUS
        if status_ok:
            if predictions[i] == 1:
                crit += 1
            else:
                crit = max(crit - 1, 0)
        if crit > max_crit:
            max_crit = crit

    return int(max_crit)


# ============================================================================
# TRAIN GLOBAL / SUBSYSTEM MODELS
# ============================================================================

print("Training global NBMs...\n")

farm_global_models = {}
farm_sub_models    = {}
event_scores       = {}

for farm in ["A", "B", "C"]:
    farm_events   = [d for d in dataset if d["farm"] == farm]
    normal_events = [d for d in farm_events if d["label"] == "normal"]
    threshold_pct = FARM_THRESHOLD_PCT[farm]

    print(f"Farm {farm}: building global NBM from {len(normal_events)} normal events (threshold pct={threshold_pct})")

    X_parts = []
    avg_cols_ref = None
    feat_cols_ref = None

    for d in normal_events:
        tr, _ = load_event(d["filepath"], farm)
        if len(tr) == 0:
            continue

        tr_eng, avg_cols_used = engineer_features(tr, farm, avg_cols_ref=avg_cols_ref)
        if avg_cols_ref is None:
            avg_cols_ref = avg_cols_used

        feat_cols = get_feature_columns(tr_eng, farm)
        if feat_cols_ref is None:
            feat_cols_ref = feat_cols

        common = [c for c in feat_cols_ref if c in tr_eng.columns]
        X = np.nan_to_num(tr_eng[common].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

        if X.shape[1] < len(feat_cols_ref):
            pad = np.zeros((len(X), len(feat_cols_ref) - X.shape[1]), dtype=np.float32)
            X = np.hstack([X, pad])

        X_parts.append(X)

        del tr, tr_eng, X
        gc.collect()

    if not X_parts:
        print(f"  No training matrix for Farm {farm}\n")
        continue

    X_train = np.vstack(X_parts).astype(np.float32)
    scaler, pca, model, n_keep, threshold, explained = train_nbm(X_train, threshold_pct=threshold_pct)
    print(f"  Global model: rows={X_train.shape[0]:,}  features={X_train.shape[1]}  PCA={n_keep}  var={explained:.1%}  thr={threshold:.4f}")

    farm_global_models[farm] = {
        "scaler": scaler,
        "pca": pca,
        "model": model,
        "n_keep": n_keep,
        "threshold": threshold,
        "feat_cols_ref": feat_cols_ref,
        "avg_cols_ref": avg_cols_ref,
    }

    del X_parts, X_train
    gc.collect()

    # Subsystem models
    farm_sub_models[farm] = {}
    if farm in SUBSYSTEM_MODEL_FARMS:
        gdict = group_dicts[farm]

        avg_cols_only = [c for c in avg_cols_ref if c in feat_cols_ref]
        group_to_cols = defaultdict(list)
        for c in avg_cols_only:
            grp = gdict.get(c, "other")
            if grp in SUBSYSTEM_TRAIN_GROUPS.get(farm, set()):
                group_to_cols[grp].append(c)

        eligible = {
            g: cols for g, cols in group_to_cols.items()
            if len(cols) >= MIN_SENSORS_FOR_SUBSYSTEM_MODEL
        }

        print(f"  Subsystem models: {list(eligible.keys())}")

        for group, sub_cols in eligible.items():
            X_sub_parts = []

            for d in normal_events:
                tr, _ = load_event(d["filepath"], farm)
                if len(tr) == 0:
                    continue

                tr_eng, _ = engineer_features(tr, farm, avg_cols_ref=avg_cols_ref)
                use_cols = [c for c in sub_cols if c in tr_eng.columns]
                if len(use_cols) < 2:
                    del tr, tr_eng
                    gc.collect()
                    continue

                Xs = np.nan_to_num(tr_eng[use_cols].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
                X_sub_parts.append(Xs)

                del tr, tr_eng, Xs
                gc.collect()

            if not X_sub_parts:
                continue

            X_sub_train = np.vstack(X_sub_parts).astype(np.float32)
            sc_s, pca_s, mod_s, nk_s, thr_s, exp_s = train_nbm(X_sub_train, threshold_pct=threshold_pct)

            farm_sub_models[farm][group] = {
                "scaler": sc_s,
                "pca": pca_s,
                "model": mod_s,
                "n_keep": nk_s,
                "threshold": thr_s,
                "feat_cols_ref": sub_cols,
            }

            print(f"    {group:22} rows={X_sub_train.shape[0]:,}  features={X_sub_train.shape[1]:3d}  PCA={nk_s:2d}  thr={thr_s:.4f}")

            del X_sub_parts, X_sub_train
            gc.collect()

    print()


# ============================================================================
# SCORE ALL EVENTS
# ============================================================================

print("Scoring all events...\n")

for d in dataset:
    farm = d["farm"]
    eid  = d["event_id"]

    if farm not in farm_global_models:
        continue

    global_cfg = farm_global_models[farm]

    _, pred_df = load_event(d["filepath"], farm)
    if len(pred_df) == 0:
        continue

    pred_eng, _ = engineer_features(pred_df, farm, avg_cols_ref=global_cfg["avg_cols_ref"])

    preds_global, scores_global = score_window(
        pred_eng,
        feat_cols_ref=global_cfg["feat_cols_ref"],
        scaler=global_cfg["scaler"],
        pca=global_cfg["pca"],
        model=global_cfg["model"],
        n_keep=global_cfg["n_keep"],
        threshold=global_cfg["threshold"],
    )

    status_ids = (
        pred_eng["status_type_id"].values
        if "status_type_id" in pred_eng.columns
        else np.zeros(len(pred_eng), dtype=int)
    )

    sub_preds_map  = {}
    sub_scores_map = {}

    for group, cfg in farm_sub_models.get(farm, {}).items():
        try:
            preds_s, scores_s = score_window(
                pred_eng,
                feat_cols_ref=cfg["feat_cols_ref"],
                scaler=cfg["scaler"],
                pca=cfg["pca"],
                model=cfg["model"],
                n_keep=cfg["n_keep"],
                threshold=cfg["threshold"],
            )
            sub_preds_map[group]  = preds_s
            sub_scores_map[group] = float(np.max(scores_s)) if len(scores_s) else 0.0
        except Exception:
            continue

    slim_cols = ["time_stamp"]
    if "status_type_id" in pred_eng.columns:
        slim_cols.append("status_type_id")
    pred_df_slim = pred_eng[slim_cols].copy()

    event_scores[eid] = {
        "event_id": eid,
        "farm": farm,
        "label": d["label"],
        "description": d["description"],
        "event_start": d["event_start"],
        "event_end": d["event_end"],
        "predictions_global": preds_global,
        "scores_global": scores_global,
        "status_ids": status_ids,
        "pred_df": pred_df_slim,
        "sub_preds_map": sub_preds_map,
        "sub_scores_map": sub_scores_map,
    }

    del pred_df, pred_eng, pred_df_slim
    gc.collect()


# ============================================================================
# CARE SCORE FUNCTIONS
# ============================================================================

def compute_fbeta(y_true, y_pred, beta=BETA):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    denom = (1 + beta**2) * tp + beta**2 * fn + fp
    if denom == 0:
        return 0.0
    return float((1 + beta**2) * tp / denom)


def compute_accuracy(y_true, y_pred):
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    if (tn + fp) == 0:
        return 0.0
    return float(tn / (tn + fp))


def compute_weighted_score(predictions, pred_df, event_start, event_end):
    n = len(predictions)
    if n == 0:
        return 0.0

    start_idx = None
    end_idx = None

    if event_start is not None and event_end is not None and "time_stamp" in pred_df.columns:
        try:
            ts = pd.to_datetime(pred_df["time_stamp"].values, errors="coerce")
            idx = np.where((ts >= pd.to_datetime(event_start)) & (ts <= pd.to_datetime(event_end)))[0]
            if len(idx) > 0:
                start_idx = int(idx[0])
                end_idx   = int(idx[-1]) + 1
        except Exception:
            pass

    if start_idx is None or end_idx is None:
        start_idx = int(n * 0.70)
        end_idx   = n

    event_len = max(1, end_idx - start_idx)
    preds_in  = predictions[start_idx:end_idx]

    weights = np.array([
        1.0 if (i / event_len) <= 0.5
        else max(0.0, 1.0 - 2.0 * ((i / event_len) - 0.5))
        for i in range(event_len)
    ], dtype=np.float32)

    if weights.sum() == 0:
        return 0.0

    return float((weights * preds_in).sum() / weights.sum())


def get_fused_predictions_for_scoring(eid, farm):
    """
    Global predictions are always present.
    Only subsystem groups explicitly allowed for alarming are fused in.
    """
    es = event_scores[eid]
    preds = es["predictions_global"].copy()

    allowed_groups = SUBSYSTEM_ALARM_GROUPS.get(farm, set())
    for group, preds_s in es.get("sub_preds_map", {}).items():
        if group in allowed_groups:
            preds = np.maximum(preds, preds_s)

    return preds


# ============================================================================
# EVENT-LEVEL RESULTS
# ============================================================================

farm_results_all = {}
care_scores = {}

for farm in ["A", "B", "C"]:
    farm_events = [d for d in dataset if d["farm"] == farm]
    farm_res = []

    for d in farm_events:
        eid = d["event_id"]
        if eid not in event_scores:
            continue

        es = event_scores[eid]
        preds_global = es["predictions_global"]
        status_ids   = es["status_ids"]

        crit_global = criticality_algorithm(preds_global, status_ids, farm)
        alarm_global = crit_global >= CRITICALITY_THRESHOLD_GLOBAL

        best_sub_group = None
        best_sub_crit  = 0
        alarm_sub      = False

        allowed_groups = SUBSYSTEM_ALARM_GROUPS.get(farm, set())
        for group, preds_s in es.get("sub_preds_map", {}).items():
            if group not in allowed_groups:
                continue
            crit_s = criticality_algorithm(preds_s, status_ids, farm)
            if crit_s > best_sub_crit:
                best_sub_crit = crit_s
                best_sub_group = group
            if crit_s >= CRITICALITY_THRESHOLD_SUB:
                alarm_sub = True

        alarm = alarm_global or alarm_sub
        max_crit = max(crit_global, best_sub_crit)

        if alarm_global:
            alarm_source = "global"
            preds_for_ws = preds_global
        elif alarm_sub and best_sub_group is not None:
            alarm_source = f"subsystem:{best_sub_group}"
            preds_for_ws = es["sub_preds_map"][best_sub_group]
        else:
            alarm_source = "none"
            preds_for_ws = preds_global

        earliness = None
        if alarm and d["label"] == "anomaly":
            earliness = compute_weighted_score(
                preds_for_ws,
                es["pred_df"],
                d["event_start"],
                d["event_end"],
            )

        sub_scores = es.get("sub_scores_map", {})
        top_subsystem = max(sub_scores, key=sub_scores.get) if sub_scores else "unknown"

        farm_res.append({
            "event_id": eid,
            "farm": farm,
            "label": d["label"],
            "event_label": d["label"],
            "description": d["description"],
            "alarm_raised": bool(alarm),
            "alarm_source": alarm_source,
            "max_criticality": int(max_crit),
            "crit_global": int(crit_global),
            "crit_subsystem": int(best_sub_crit),
            "anomaly_rate": round(float(preds_global.mean()), 4),
            "earliness": round(float(earliness), 4) if earliness is not None else None,
            "n_pred_rows": int(len(preds_global)),
            "event_start": d["event_start"],
            "event_end": d["event_end"],
            "top_subsystem": top_subsystem,
            "subsystem_scores": sub_scores,
        })

    farm_results_all[farm] = farm_res


# ============================================================================
# FARM-LEVEL CARE SCORE
# ============================================================================

def compute_care_score(farm_results, farm):
    anomaly_r = [r for r in farm_results if r["label"] == "anomaly"]
    normal_r  = [r for r in farm_results if r["label"] == "normal"]

    # Coverage
    all_y_true = []
    all_y_pred = []

    for r in anomaly_r:
        eid = r["event_id"]
        if eid not in event_scores:
            continue

        es   = event_scores[eid]
        pred = get_fused_predictions_for_scoring(eid, farm)
        prd  = es["pred_df"]
        n    = len(pred)

        y_true = np.zeros(n, dtype=int)

        try:
            ts = pd.to_datetime(prd["time_stamp"].values, errors="coerce")
            mask = (ts >= pd.to_datetime(r["event_start"])) & (ts <= pd.to_datetime(r["event_end"]))
            y_true[mask] = 1
        except Exception:
            y_true[int(n * 0.70):] = 1

        if farm != "A":
            mask_normal = np.array([int(s) in NORMAL_STATUS for s in es["status_ids"]])
            y_true = y_true[mask_normal]
            pred   = pred[mask_normal]

        all_y_true.extend(y_true.tolist())
        all_y_pred.extend(pred.tolist())

    coverage = compute_fbeta(np.array(all_y_true), np.array(all_y_pred)) if all_y_true else 0.0

    # Accuracy
    acc_scores = []
    for r in normal_r:
        eid = r["event_id"]
        if eid not in event_scores:
            continue

        es   = event_scores[eid]
        pred = get_fused_predictions_for_scoring(eid, farm)
        y_true = np.zeros(len(pred), dtype=int)

        if farm != "A":
            mask_normal = np.array([int(s) in NORMAL_STATUS for s in es["status_ids"]])
            pred = pred[mask_normal]
            y_true = y_true[mask_normal]

        acc_scores.append(compute_accuracy(y_true, pred))

    accuracy = float(np.mean(acc_scores)) if acc_scores else 0.0

    # Reliability
    ev_true = []
    ev_pred = []

    for r in farm_results:
        eid = r["event_id"]
        if eid not in event_scores:
            continue

        es = event_scores[eid]

        preds_fused = get_fused_predictions_for_scoring(eid, farm)
        crit_global_or_fused = criticality_algorithm(preds_fused, es["status_ids"], farm)
        alarm = int(crit_global_or_fused >= CRITICALITY_THRESHOLD_GLOBAL)

        # also allow subsystem event alarm only if allowed
        allowed_groups = SUBSYSTEM_ALARM_GROUPS.get(farm, set())
        for group, preds_s in es.get("sub_preds_map", {}).items():
            if group not in allowed_groups:
                continue
            crit_s = criticality_algorithm(preds_s, es["status_ids"], farm)
            if crit_s >= CRITICALITY_THRESHOLD_SUB:
                alarm = 1
                break

        ev_true.append(1 if r["label"] == "anomaly" else 0)
        ev_pred.append(alarm)

    reliability = compute_fbeta(np.array(ev_true), np.array(ev_pred)) if ev_true else 0.0

    # Earliness
    ws_scores = []
    for r in anomaly_r:
        if not r["alarm_raised"]:
            continue
        eid = r["event_id"]
        es = event_scores[eid]

        preds = get_fused_predictions_for_scoring(eid, farm)
        ws = compute_weighted_score(preds, es["pred_df"], r["event_start"], r["event_end"])
        ws_scores.append(ws)

    earliness = float(np.mean(ws_scores)) if ws_scores else 0.0

    n_detected = sum(1 for r in anomaly_r if r["alarm_raised"])
    n_false = sum(1 for r in normal_r if r["alarm_raised"])

    if n_detected == 0:
        care = 0.0
    elif accuracy < 0.5:
        care = accuracy
    else:
        care = (coverage + earliness + reliability + 2 * accuracy) / 5.0

    return {
        "coverage": round(float(coverage), 3),
        "accuracy": round(float(accuracy), 3),
        "reliability": round(float(reliability), 3),
        "earliness": round(float(earliness), 3),
        "care": round(float(care), 3),
        "n_anomaly": len(anomaly_r),
        "n_normal": len(normal_r),
        "n_detected": n_detected,
        "n_false_alarms": n_false,
    }


for farm in ["A", "B", "C"]:
    care_scores[farm] = compute_care_score(farm_results_all.get(farm, []), farm)


# ============================================================================
# PLOTS
# ============================================================================

def plot_event_balance():
    farms = ["A", "B", "C"]
    an = [sum(1 for d in dataset if d["farm"] == f and d["label"] == "anomaly") for f in farms]
    no = [sum(1 for d in dataset if d["farm"] == f and d["label"] == "normal")  for f in farms]

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(farms))
    w = 0.35
    b1 = ax.bar(x - w/2, an, w, label="Anomaly", alpha=0.85)
    b2 = ax.bar(x + w/2, no, w, label="Normal", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f"Farm {f}" for f in farms])
    ax.set_ylabel("Number of events")
    ax.set_title("Dataset balance — events per farm", fontweight="bold")
    ax.bar_label(b1, padding=2)
    ax.bar_label(b2, padding=2)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "01_event_balance.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_care_scores():
    farms = ["A", "B", "C"]
    subs = ["coverage", "accuracy", "reliability", "earliness", "care"]
    labels = ["Coverage", "Accuracy", "Reliability", "Earliness", "CARE"]

    fig, axes = plt.subplots(1, 5, figsize=(18, 4.8))
    for i, (sub, lbl) in enumerate(zip(subs, labels)):
        vals = [care_scores[f][sub] for f in farms]
        bars = axes[i].bar([f"Farm {f}" for f in farms], vals, alpha=0.85)
        axes[i].set_ylim(0, 1.1)
        axes[i].set_title(lbl, fontweight="bold", fontsize=10)
        axes[i].bar_label(bars, fmt="%.3f", padding=2, fontsize=8)
        axes[i].grid(axis="y", alpha=0.3)
        if sub == "care":
            axes[i].axhline(0.66, color="orange", linestyle="--", linewidth=1.2, alpha=0.8)

    fig.suptitle("CARE score breakdown", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "02_care_scores.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_criticality_per_farm(farm):
    rows = farm_results_all.get(farm, [])
    if not rows:
        return

    rows = sorted(rows, key=lambda r: int(r["event_id"]))
    labels = [r["event_id"] for r in rows]
    vals   = [r["max_criticality"] for r in rows]
    cols   = ["#d62728" if r["label"] == "anomaly" else "#2ca02c" for r in rows]

    fig, ax = plt.subplots(figsize=(max(10, len(rows) * 0.55), 5))
    bars = ax.bar(range(len(rows)), vals, color=cols, alpha=0.85)
    ax.axhline(CRITICALITY_THRESHOLD_GLOBAL, color="black", linestyle="--", linewidth=1.4)
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Max criticality")
    ax.set_title(f"Farm {farm} — event criticality", fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f"03_criticality_farm{farm}.png"), dpi=140, bbox_inches="tight")
    plt.close()


print("Generating plots...\n")
plot_event_balance()
plot_care_scores()
for farm in ["A", "B", "C"]:
    plot_criticality_per_farm(farm)


# ============================================================================
# SAVE OUTPUTS
# ============================================================================

def json_safe(obj):
    if isinstance(obj, dict):
        return {k: json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_safe(v) for v in obj]
    if isinstance(obj, set):
        return sorted(list(obj))
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return str(obj)
    if pd.isna(obj) if not isinstance(obj, (list, dict, tuple, set, np.ndarray)) else False:
        return None
    return obj


results_json = {
    "timestamp": datetime.now().isoformat(),
    "script": "script_1_anomaly_detection_clean_improved",
    "base_path": BASE_PATH,
    "parameters": {
        "beta": BETA,
        "criticality_threshold_global": CRITICALITY_THRESHOLD_GLOBAL,
        "criticality_threshold_sub": CRITICALITY_THRESHOLD_SUB,
        "smoothing_window": SMOOTHING_WINDOW,
        "if_contamination": IF_CONTAMINATION,
        "if_n_estimators": IF_N_ESTIMATORS,
        "pca_variance_target": PCA_VARIANCE_TARGET,
        "farm_threshold_pct": FARM_THRESHOLD_PCT,
        "subsystem_alarm_groups": json_safe(SUBSYSTEM_ALARM_GROUPS),
    },
    "care_scores": care_scores,
    "event_results": [r for farm in ["A", "B", "C"] for r in farm_results_all.get(farm, [])],
}

json_path = os.path.join(OUTPUT_DIR, "results_stage1_clean.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_safe(results_json), f, indent=2)

events_csv_path = os.path.join(OUTPUT_DIR, "event_results_stage1_clean.csv")
pd.DataFrame(results_json["event_results"]).drop(columns=["subsystem_scores"], errors="ignore").to_csv(events_csv_path, index=False)

care_csv_path = os.path.join(REPORT_DIR, "care_score_summary_clean.csv")
pd.DataFrame([{"farm": farm, **care_scores[farm]} for farm in ["A", "B", "C"]]).to_csv(care_csv_path, index=False)

print("CARE score breakdown:")
for farm in ["A", "B", "C"]:
    cs = care_scores[farm]
    print(
        f"  Farm {farm}: CARE={cs['care']:.3f}  "
        f"(cov={cs['coverage']:.3f}, acc={cs['accuracy']:.3f}, "
        f"rel={cs['reliability']:.3f}, ear={cs['earliness']:.3f})  "
        f"detected={cs['n_detected']}/{cs['n_anomaly']}  "
        f"false={cs['n_false_alarms']}/{cs['n_normal']}"
    )

print("\nSaved outputs:")
print(f"  {json_path}")
print(f"  {events_csv_path}")
print(f"  {care_csv_path}")

print("\n" + "=" * 80)
print("SCRIPT 1 COMPLETE")
print("=" * 80)
for farm in ["A", "B", "C"]:
    cs = care_scores[farm]
    print(f"Farm {farm}: CARE={cs['care']:.3f} | cov={cs['coverage']:.3f} acc={cs['accuracy']:.3f} rel={cs['reliability']:.3f} ear={cs['earliness']:.3f}")

SCRIPT 1 — CARE ANOMALY DETECTION (FULL CLEAN REBUILD, IMPROVED)
Base directory: E:\project\CARE_To_Compare

Farm A registry:
  Column mappings: 81
  electrical             sensors= 14  cols= 29  nbm= 29
  temperature            sensors= 11  cols= 11  nbm= 11
  generator_other        sensors=  7  cols= 10  nbm= 10
  angle                  sensors=  4  cols=  7  nbm=  0
  generator_stator       sensors=  3  cols=  3  nbm=  3
  transformer            sensors=  3  cols=  3  nbm=  3
  other                  sensors=  2  cols=  5  nbm=  5
  environment            sensors=  2  cols=  2  nbm=  2
  generator_bearing      sensors=  2  cols=  2  nbm=  2
  control                sensors=  1  cols=  4  nbm=  4
  cooling_water          sensors=  1  cols=  1  nbm=  1
  gearbox_bearing        sensors=  1  cols=  1  nbm=  1

Farm B registry:
  Column mappings: 252
  electrical             sensors= 16  cols= 64  nbm= 64
  transformer            sensors=  9  cols= 36  nbm= 36
  pitch                  se

In [21]:
# -*- coding: utf-8 -*-
"""
================================================================================
SCRIPT 2 — CARE FAULT CLASSIFICATION (CLEAN REBUILD)
================================================================================
Purpose
-------
Stage 2 multiclass fault classification for the CARE benchmark.

This script:
  - uses exact sensor names from feature_description.csv
  - performs LOEO (Leave-One-Event-Out) per farm
  - trains on prediction windows of training events
  - optionally adds capped status-4 rows from training windows
  - applies fault-type-aware sensor group selection
  - uses XGBoost for multiclass classification
  - reports detection rate, classification accuracy, and horizon

Important methodological note
-----------------------------
This stage is diagnostic, not anomaly-detection.
It assumes Stage 1 has already indicated abnormal behaviour.
The classifier therefore learns fault signatures from fault events,
not normal-vs-fault separation.

Author: Peter
Dataset root: E:\\project\\CARE_To_Compare
"""

# ============================================================================
# IMPORTS
# ============================================================================

import os
import gc
import re
import json
import warnings
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import xgboost as xgb


# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_PATH = r"E:\project\CARE_To_Compare"

EVENT_INFO_PATH = os.path.join(BASE_PATH, "df_all__1_.csv")

FARMS = {
    "A": os.path.join(BASE_PATH, "Wind Farm A", "datasets"),
    "B": os.path.join(BASE_PATH, "Wind Farm B", "datasets"),
    "C": os.path.join(BASE_PATH, "Wind Farm C", "datasets"),
}

FEATURE_DESC = {
    "A": os.path.join(BASE_PATH, "Wind Farm A", "feature_description.csv"),
    "B": os.path.join(BASE_PATH, "Wind Farm B", "feature_description.csv"),
    "C": os.path.join(BASE_PATH, "Wind Farm C", "feature_description.csv"),
}

OUTPUT_DIR = os.path.join(BASE_PATH, "stage2_clean_outputs")
PLOT_DIR   = os.path.join(OUTPUT_DIR, "plots")
REPORT_DIR = os.path.join(OUTPUT_DIR, "reports")

for d in [OUTPUT_DIR, PLOT_DIR, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)

CSV_SEP = ";"
FARM_NAME_MAP = {"Farm A": "A", "Farm B": "B", "Farm C": "C"}
NORMAL_STATUS = {0, 2}

META_COLS = {"time_stamp", "asset_id", "id", "train_test", "status_type_id"}

# Detection rule
CONSECUTIVE_WINDOW = 5
CONSECUTIVE_WINDOW_SHORT = 3
SHORT_WINDOW_THRESHOLD = 200   # timesteps

# Confidence threshold
CONFIDENCE_THRESHOLD = 0.40

# Oversampling
MIN_EXAMPLES_FOR_CLASS = 5
OVERSAMPLE_NOISE_STD   = 0.015
MAX_CLASS_WEIGHT       = 8.0

# Train data mix
USE_STATUS4_ROWS = True
MAX_STATUS4_MULTIPLIER = 2   # cap status-4 rows to 2x prediction-window length

# XGBoost parameters
XGB_PARAMS_SMALL = dict(
    max_depth=4,
    n_estimators=400,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.7,
    colsample_bylevel=0.8,
    reg_lambda=2.5,
    reg_alpha=0.15,
    min_child_weight=2,
    gamma=0.1,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

# Limit raw avg columns per fold before engineering
MAX_RAW_AVG_COLS_PER_FOLD = 120


# ============================================================================
# SUBSYSTEM RULES
# ============================================================================

SUBSYSTEM_RULES = [
    (10,  "gearbox_bearing",   ["planetary bearing", "gearbox bearing",
                                "bearing on high speed shaft"]),
    (11,  "gearbox_oil",       ["gearbox oil", "gear oil", "oil inlet temperature",
                                "oil pressure", "oil level", "oil tank",
                                "oil temperature"]),
    (12,  "gearbox_other",     ["gearbox rotational", "gearbox speed",
                                "gearbox main shaft", "gearbox"]),
    (20,  "generator_bearing", ["generator bearing"]),
    (21,  "generator_stator",  ["stator winding", "stator"]),
    (22,  "generator_cooling", ["generator cooling", "cooling air",
                                "cooling water temp. generator",
                                "generator temperature", "cooling air inlet"]),
    (23,  "generator_other",   ["generator acceleration", "generator angle",
                                "generator rpm", "generator converter",
                                "generator rms", "generator"]),
    (30,  "rotor_bearing",     ["rotor bearing", "axial bearing",
                                "rotor bearing inner ring"]),
    (40,  "hydraulic",         ["hydraulic pressure", "hydraulic oil",
                                "hydraulic pump", "hydraulic tank",
                                "hydraulic group", "aggregate system pressure",
                                "azimuth circuit", "azimuth memory",
                                "rotor brake", "locking memory",
                                "pollution indicator", "oil container"]),
    (50,  "pitch",             ["pitch angle", "pitch motor", "pitch axis",
                                "position motor axis", "position rotor blade",
                                "motor temperature kty", "motor temperature pt",
                                "axis 1 motor", "axis 2 motor", "axis 3 motor",
                                "axis 1 voltage", "axis 2 voltage", "axis 3 voltage",
                                "battery charge", "battery discharge",
                                "rpm motor axis", "min pitch",
                                "motor current axis", "motor temperature axis"]),
    (60,  "vibration",         ["vibration", "drive train vibration",
                                "tower vibration"]),
    (70,  "transformer",       ["transformer", "transformator",
                                "hv transformer", "transformer cell"]),
    (80,  "electrical",        ["voltage", "current", "frequency",
                                "active power", "reactive power",
                                "apparent power", "grid power", "grid voltage",
                                "grid frequency", "available power",
                                "inverter torque", "cable load",
                                "dc link voltage", "internal consumption"]),
    (90,  "wind",              ["wind speed", "wind direction", "anemometer",
                                "estimated windspeed", "absolute wind",
                                "relative wind"]),
    (100, "environment",       ["ambient temperature", "outside temperature",
                                "nacelle temperature", "nacelle external",
                                "hub temperature", "board temperature",
                                "electrical cabinet", "converter room",
                                "control box temperature", "nacelle outside",
                                "platform temperature"]),
    (110, "cooling_water",     ["water temperature", "water pressure",
                                "water flow", "water conductivity",
                                "water cooler", "cooling water",
                                "flow meter stator"]),
    (120, "rotor",             ["rotor speed", "rotor rpm",
                                "nacelle direction", "yaw"]),
]


def assign_group(description):
    desc_lower = str(description).lower()
    for _, group, keywords in sorted(SUBSYSTEM_RULES, key=lambda x: x[0]):
        if any(kw in desc_lower for kw in keywords):
            return group
    return "other"


# ============================================================================
# FAULT LABELS
# ============================================================================

FAULT_LABELS = {
    "A": {
        68: "transformer",
        22: "hydraulic",
        72: "gearbox",
        73: "hydraulic",
        0:  "generator_bearing",
        26: "hydraulic",
        40: "generator_bearing",
        42: "hydraulic",
        10: "gearbox",
        45: "hydraulic",
        84: "hydraulic",
        51: "gearbox",
    },
    "B": {
        34: "transformer",
        7:  "transformer",
        53: "rotor_bearing",
        27: "rotor_bearing",
        19: "transformer",
        77: "rotor_bearing",
    },
    "C": {
        55: "pitch_electrical",
        81: "converter",
        47: "hydraulic",
        12: "gearbox_oil",
        4:  "pitch_electrical",
        18: "rotor_brake",
        28: "pitch_electrical",
        39: "gearbox_oil",
        66: "pitch_electrical",
        15: "pitch_electrical",
        78: "rotor_brake",
        79: "communication",
        30: "pitch_electrical",
        33: "pitch_mechanical",
        11: "pitch_electrical",
        44: "cooling_water",
        49: "gearbox_oil",
        31: "communication",
        67: "transformer",
        9:  "yaw",
        91: "pitch_electrical",
        5:  "electrical_internal",
        90: "communication",
        70: "pitch_electrical",
        35: "converter",
        16: "pitch_electrical",
        76: "pitch_electrical",
    },
}


# ============================================================================
# SENSOR GROUP SELECTION BY FAULT TYPE
# ============================================================================

FAULT_SENSOR_GROUPS = {
    "transformer":         ["transformer", "electrical"],
    "gearbox":             ["gearbox_bearing", "gearbox_oil", "gearbox_other", "vibration"],
    "gearbox_oil":         ["gearbox_oil", "gearbox_other"],
    "generator_bearing":   ["generator_bearing", "generator_stator",
                            "generator_cooling", "generator_other", "electrical"],
    "hydraulic":           ["hydraulic"],
    "rotor_bearing":       ["rotor_bearing", "vibration"],
    "pitch_electrical":    ["pitch", "electrical"],
    "pitch_mechanical":    ["pitch"],
    "converter":           ["electrical", "vibration"],
    "communication":       ["electrical"],
    "rotor_brake":         ["hydraulic", "rotor_bearing"],
    "yaw":                 ["rotor", "electrical"],
    "cooling_water":       ["cooling_water", "generator_cooling"],
    "electrical_internal": ["electrical"],
    "normal":              [],
}

OPERATIONAL_GROUPS = {"environment", "wind", "rotor", "electrical"}


# ============================================================================
# SENSOR RENAMING / REGISTRY
# ============================================================================

STAT_MAP = {
    "average": "avg",
    "std_dev": "std",
    "maximum": "max",
    "minimum": "min",
}


def _clean_text(text, is_unit=False):
    if pd.isna(text) or str(text).strip().lower() in ("", "nan", "-", "none"):
        return ""
    t = str(text).strip()

    replacements = [
        ("���C", "degC"), ("°C", "degC"), ("ºC", "degC"), ("½C", "degC"),
        ("Celsius", "degC"), ("���", "deg"), ("°", "deg"), ("º", "deg"),
        ("½", ""), ("/", "_per_"), ("%", "pct"), (" ", ""), ("^", ""), (".", "")
    ]
    for old, new in replacements:
        t = t.replace(old, new)

    t = re.sub(r"[^\x00-\x7F]+", "", t)

    if not is_unit:
        t = re.sub(r"[^\w\s]", " ", t)
        t = re.sub(r"\s+", "_", t.strip())
        t = re.sub(r"_+", "_", t).strip("_")
    else:
        t = re.sub(r"[^\w]", "", t)

    return t


def build_sensor_registry(farm_id):
    path = FEATURE_DESC[farm_id]
    df = None

    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(path, sep=";", encoding=enc)
            break
        except UnicodeDecodeError:
            continue

    if df is None:
        raise RuntimeError(f"Could not read feature description for Farm {farm_id}")

    rename_dict = {}
    counter_new = set()
    angle_new   = set()
    group_dict  = {}
    rows = []
    seen = {}

    for _, row in df.iterrows():
        sid        = str(row["sensor_name"]).strip()
        desc_raw   = str(row.get("description", "")).strip()
        unit_raw   = str(row.get("unit", "")).strip()
        is_counter = bool(row.get("is_counter", False))
        is_angle   = bool(row.get("is_angle", False))

        desc = _clean_text(desc_raw, is_unit=False)
        unit = _clean_text(unit_raw, is_unit=True)
        group = assign_group(desc_raw)

        stats_raw = str(row.get("statistics_type", "")).strip()
        if stats_raw.lower() in ("", "nan"):
            stats = []
        else:
            stats = [s.strip() for s in stats_raw.split(",") if s.strip()]

        entries = []
        if stats:
            for s in stats:
                sc = STAT_MAP.get(s, s)
                entries.append((f"{sid}_{sc}", sc))
        else:
            entries.append((sid, ""))

        for old_col, sc in entries:
            parts = [desc]
            if unit:
                parts.append(unit)
            if sc:
                parts.append(sc)
            new_col = "_".join(parts)

            if new_col in seen and seen[new_col] != old_col:
                new_col = f"{new_col}_{sid}"
            seen[new_col] = old_col

            rename_dict[old_col] = new_col
            group_dict[new_col]  = group

            if is_counter:
                counter_new.add(new_col)
            if is_angle:
                angle_new.add(new_col)

            rows.append({
                "farm": farm_id,
                "sensor_id": sid,
                "old_col": old_col,
                "new_col": new_col,
                "description": desc_raw,
                "unit": unit_raw,
                "stat": sc,
                "group": group,
                "is_counter": is_counter,
                "is_angle": is_angle,
                "use_in_model": not (is_counter or is_angle),
            })

    registry_df = pd.DataFrame(rows)
    return rename_dict, counter_new, angle_new, group_dict, registry_df


print("=" * 80)
print("SCRIPT 2 — CARE FAULT CLASSIFICATION (CLEAN REBUILD)")
print("=" * 80)
print(f"Base directory: {BASE_PATH}\n")

rename_dicts = {}
counter_sets = {}
angle_sets   = {}
group_dicts  = {}
registries   = {}

for farm in ["A", "B", "C"]:
    rd, co, an, gd, reg = build_sensor_registry(farm)
    rename_dicts[farm] = rd
    counter_sets[farm] = co
    angle_sets[farm]   = an
    group_dicts[farm]  = gd
    registries[farm]   = reg

    print(f"Farm {farm} registry:")
    print(f"  Column mappings: {len(rd)}")

    summary = (
        reg.groupby("group")
        .agg(
            sensors=("sensor_id", "nunique"),
            cols=("new_col", "count"),
            model_cols=("use_in_model", "sum")
        )
        .sort_values(["sensors", "cols"], ascending=False)
    )
    for group, row in summary.head(12).iterrows():
        print(f"  {group:22} sensors={int(row['sensors']):3d}  cols={int(row['cols']):3d}  model={int(row['model_cols']):3d}")
    print()

sensor_registry_path = os.path.join(REPORT_DIR, "sensor_registry_exact_stage2.csv")
pd.concat(registries.values(), ignore_index=True).to_csv(sensor_registry_path, index=False)
print(f"Saved sensor registry: {sensor_registry_path}\n")


# ============================================================================
# EVENT METADATA
# ============================================================================

event_info = pd.read_csv(EVENT_INFO_PATH)
event_info["farm"] = event_info["Farm"].map(FARM_NAME_MAP)
event_info["event_id"] = event_info["event_id"].astype(str)
event_info["event_label"] = event_info["event_label"].astype(str).str.strip()
event_info["event_description"] = event_info["event_description"].fillna("normal").astype(str)

dataset = []
for _, row in event_info.iterrows():
    farm = row["farm"]
    if pd.isna(farm):
        continue

    eid = row["event_id"]
    eid_int = int(eid) if eid.isdigit() else -1
    label = row["event_label"]
    filepath = os.path.join(FARMS[farm], f"{eid}.csv")
    if not os.path.exists(filepath):
        continue

    fault_type = FAULT_LABELS.get(farm, {}).get(eid_int, "normal")
    if label == "normal":
        fault_type = "normal"

    dataset.append({
        "event_id": eid,
        "farm": farm,
        "label": label,
        "fault_type": fault_type,
        "filepath": filepath,
        "description": row["event_description"],
    })

print("Event counts:")
for farm in ["A", "B", "C"]:
    fe = [d for d in dataset if d["farm"] == farm]
    n_total = len(fe)
    n_anom  = sum(1 for d in fe if d["label"] == "anomaly")
    n_norm  = sum(1 for d in fe if d["label"] == "normal")
    print(f"  Farm {farm}: total={n_total}  anomaly={n_anom}  normal={n_norm}")
print()


# ============================================================================
# DATA LOADING
# ============================================================================

def _handle_zero_fills(df, farm_id, numeric_cols):
    excluded = counter_sets[farm_id] | angle_sets[farm_id]
    for col in numeric_cols:
        if col in excluded:
            continue
        vals = df[col].values.copy()
        try:
            vals = vals.astype(np.float32)
        except Exception:
            continue

        n = len(vals)
        i = 0
        while i < n:
            if vals[i] == 0.0:
                s = i
                while i < n and vals[i] == 0.0:
                    i += 1
                if i - s >= 6:
                    vals[s:i] = np.nan
            else:
                i += 1
        df[col] = vals
    return df


def load_event_selected(filepath, farm_id, selected_cols=None, chunk_size=10000):
    """
    Returns:
        pred_df       : prediction window
        status4_df    : status_type_id == 4 rows from training window
    """
    rdict = rename_dicts[farm_id]
    chunks = []

    for chunk in pd.read_csv(filepath, sep=CSV_SEP, chunksize=chunk_size,
                             low_memory=False, encoding="latin-1"):
        chunk = chunk.rename(columns=rdict)

        if selected_cols is not None:
            keep = [c for c in chunk.columns if c in META_COLS or c in selected_cols]
            chunk = chunk[[c for c in keep if c in chunk.columns]]

        num_cols = [c for c in chunk.columns if c not in META_COLS and pd.api.types.is_numeric_dtype(chunk[c])]
        chunk = _handle_zero_fills(chunk, farm_id, num_cols)

        if num_cols:
            chunk[num_cols] = chunk[num_cols].astype(np.float32)

        chunks.append(chunk)

    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()

    pred_df = df[df["train_test"] == "prediction"].copy()

    status4_df = pd.DataFrame()
    if "status_type_id" in df.columns:
        status4_df = df[(df["train_test"] == "train") & (df["status_type_id"] == 4)].copy()

    del df
    gc.collect()
    return pred_df, status4_df


# ============================================================================
# SENSOR SELECTION
# ============================================================================

def get_selected_columns(fault_types, farm_id):
    gdict = group_dicts[farm_id]
    excluded = counter_sets[farm_id] | angle_sets[farm_id]

    target_groups = set(OPERATIONAL_GROUPS)
    for ft in fault_types:
        for grp in FAULT_SENSOR_GROUPS.get(ft, []):
            target_groups.add(grp)

    selected = []
    for col, grp in gdict.items():
        if grp in target_groups and col not in excluded and "avg" in col.lower():
            selected.append(col)

    selected = list(dict.fromkeys(selected))

    # cap by group-balanced selection if too many
    if len(selected) > MAX_RAW_AVG_COLS_PER_FOLD:
        grouped = defaultdict(list)
        for c in selected:
            grouped[gdict.get(c, "other")].append(c)

        reduced = []
        per_group_cap = max(4, MAX_RAW_AVG_COLS_PER_FOLD // max(1, len(grouped)))
        for grp, cols in grouped.items():
            reduced.extend(cols[:per_group_cap])

        selected = reduced[:MAX_RAW_AVG_COLS_PER_FOLD]

    return set(selected), sorted(target_groups)


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================

def engineer_features(df, farm_id):
    excluded = counter_sets[farm_id] | angle_sets[farm_id]
    df = df.copy()

    avg_cols = [
        c for c in df.columns
        if c not in META_COLS
        and c not in excluded
        and "avg" in c.lower()
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    new_feats = {}

    # rolling stats
    for col in avg_cols:
        s = df[col].astype(np.float32)
        for w, wn in [(6, "1hr"), (18, "3hr"), (36, "6hr")]:
            new_feats[f"{col}_roll{wn}_mean"] = s.rolling(w, min_periods=1).mean().astype(np.float32)
            new_feats[f"{col}_roll{wn}_std"]  = s.rolling(w, min_periods=1).std().fillna(0).astype(np.float32)

    # temperature gradients
    temp_cols = [c for c in avg_cols if ("temp" in c.lower() or "degc" in c.lower())]
    for col in temp_cols:
        s = df[col].astype(np.float32)
        new_feats[f"{col}_diff1"]  = s.diff(1).fillna(0).astype(np.float32)
        new_feats[f"{col}_diff6"]  = s.diff(6).fillna(0).astype(np.float32)
        new_feats[f"{col}_diff18"] = s.diff(18).fillna(0).astype(np.float32)
        new_feats[f"{col}_accel6"] = s.diff(6).diff(6).fillna(0).astype(np.float32)

    # current phase imbalance
    curr_cols = sorted([
        c for c in df.columns
        if "current" in c.lower()
        and "phase" in c.lower()
        and "avg" in c.lower()
        and pd.api.types.is_numeric_dtype(df[c])
    ])
    if len(curr_cols) >= 3:
        mat = df[curr_cols[:3]].values.astype(np.float32)
        new_feats["generator_current_imbalance"] = np.std(mat, axis=1).astype(np.float32)
        new_feats["generator_current_range"]     = np.ptp(mat, axis=1).astype(np.float32)
        mean_c = np.mean(mat, axis=1) + 1e-6
        new_feats["generator_current_cv"]        = (np.std(mat, axis=1) / mean_c).astype(np.float32)

    # stator imbalance
    stator_cols = sorted([
        c for c in df.columns
        if "stator" in c.lower()
        and "avg" in c.lower()
        and pd.api.types.is_numeric_dtype(df[c])
    ])
    if len(stator_cols) >= 3:
        mat = df[stator_cols[:3]].values.astype(np.float32)
        new_feats["generator_stator_temp_imbalance"] = np.std(mat, axis=1).astype(np.float32)
        new_feats["generator_stator_temp_range"]     = np.ptp(mat, axis=1).astype(np.float32)

    # transformer imbalance
    trans_cols = sorted([
        c for c in df.columns
        if "transformer" in c.lower()
        and ("temp" in c.lower() or "degc" in c.lower())
        and "avg" in c.lower()
    ])
    if len(trans_cols) >= 3:
        mat = df[trans_cols[:3]].values.astype(np.float32)
        new_feats["transformer_temp_imbalance"] = np.std(mat, axis=1).astype(np.float32)
        new_feats["transformer_temp_range"]     = np.ptp(mat, axis=1).astype(np.float32)

    # planetary bearing spread
    gb_cols = sorted([
        c for c in df.columns
        if "planetary" in c.lower()
        and ("temp" in c.lower() or "degc" in c.lower())
        and "avg" in c.lower()
    ])
    if len(gb_cols) >= 4:
        mat = df[gb_cols].values.astype(np.float32)
        new_feats["planetary_bearing_temp_mean"]  = np.mean(mat, axis=1).astype(np.float32)
        new_feats["planetary_bearing_temp_std"]   = np.std(mat, axis=1).astype(np.float32)
        new_feats["planetary_bearing_temp_range"] = np.ptp(mat, axis=1).astype(np.float32)

    # thermal normalisation
    nacelle_col = next((c for c in df.columns if "nacelle" in c.lower() and "temp" in c.lower() and "avg" in c.lower()), None)
    ambient_col = next((c for c in df.columns if "ambient" in c.lower() and "avg" in c.lower()), None)
    ref_col = nacelle_col or ambient_col

    if ref_col is not None:
        ref = df[ref_col].astype(np.float32)
        for col in avg_cols:
            if any(k in col.lower() for k in ["gearbox_oil", "generator_bearing", "hydraulic", "transformer"]):
                new_feats[f"{col}_normalised"] = (df[col].astype(np.float32) - ref).astype(np.float32)

    # physics-informed features
    power_col = next((c for c in df.columns if "active_power" in c.lower() and "avg" in c.lower()), None)
    wind_col  = next((c for c in df.columns if "wind" in c.lower() and "speed" in c.lower() and "avg" in c.lower()), None)
    rotor_col = next((c for c in df.columns if (("rotor" in c.lower()) or ("generator" in c.lower()))
                      and (("rpm" in c.lower()) or ("speed" in c.lower()))
                      and "avg" in c.lower()), None)
    gbox_oil_col = next((c for c in df.columns if "gearbox_oil" in c.lower() and "avg" in c.lower()), None)
    hydr_col = next((c for c in df.columns if "hydraulic" in c.lower() and "temp" in c.lower() and "avg" in c.lower()), None)
    pitch_col = next((c for c in df.columns if "pitch" in c.lower() and "avg" in c.lower()), None)

    if power_col and wind_col:
        wind = df[wind_col].astype(np.float32)
        pwr  = df[power_col].astype(np.float32)
        ideal = 0.5 * wind**3
        new_feats["power_curve_deviation"] = (pwr - ideal).astype(np.float32)
        new_feats["power_fluctuation_std"] = pwr.rolling(18, min_periods=1).std().fillna(0).astype(np.float32)

    if rotor_col and power_col:
        rpm = df[rotor_col].astype(np.float32)
        pwr = df[power_col].astype(np.float32)
        new_feats["generator_efficiency"] = (pwr / (rpm + 1.0)).astype(np.float32)

    if gbox_oil_col and rotor_col:
        rpm = df[rotor_col].astype(np.float32)
        new_feats["gearbox_temp_per_rpm"] = (df[gbox_oil_col].astype(np.float32) / (rpm + 1.0)).astype(np.float32)

    if gbox_oil_col and ref_col:
        new_feats["gearbox_oil_temp_rise"] = (df[gbox_oil_col].astype(np.float32) - df[ref_col].astype(np.float32)).astype(np.float32)

    if hydr_col and pitch_col:
        new_feats["hydraulic_stress"] = (
            df[hydr_col].astype(np.float32) * df[pitch_col].astype(np.float32).abs()
        ).astype(np.float32)

    out = pd.concat([df, pd.DataFrame(new_feats, index=df.index)], axis=1)

    feat_cols = [
        c for c in out.columns
        if c not in META_COLS
        and c not in excluded
        and pd.api.types.is_numeric_dtype(out[c])
    ]

    out[feat_cols] = out[feat_cols].fillna(0).astype(np.float32)
    return out, feat_cols


# ============================================================================
# DETECTION RULES
# ============================================================================

def detect_failure(predictions, true_label, timestamps):
    n = len(predictions)
    if n == 0:
        return False, None

    w = CONSECUTIVE_WINDOW_SHORT if n < SHORT_WINDOW_THRESHOLD else CONSECUTIVE_WINDOW

    for i in range(n - w + 1):
        if np.all(predictions[i:i+w] == true_label):
            if timestamps is not None and len(timestamps) > 0:
                det_ts = pd.Timestamp(timestamps[i])
                end_ts = pd.Timestamp(timestamps[-1])
                horizon_days = (end_ts - det_ts).total_seconds() / 86400
            else:
                horizon_days = (n - i) * (10 / 60 / 24)
            return True, float(horizon_days)

    return False, None


def detect_failure_window(predictions, true_label, window_frac=0.10):
    n = len(predictions)
    if n == 0:
        return False
    start = int(n * (1 - window_frac))
    return bool(np.any(predictions[start:] == true_label))


# ============================================================================
# CLASS IMBALANCE HANDLING
# ============================================================================

def oversample_minority_classes(X, y, min_examples=MIN_EXAMPLES_FOR_CLASS, noise_std=OVERSAMPLE_NOISE_STD):
    unique_cls, counts = np.unique(y, return_counts=True)
    X_aug = [X]
    y_aug = [y]

    feat_std = np.std(X, axis=0) + 1e-8
    rng = np.random.default_rng(42)

    for cls, cnt in zip(unique_cls, counts):
        if cnt < min_examples:
            needed = min_examples - cnt
            src_idx = np.where(y == cls)[0]
            rep_idx = src_idx[np.arange(needed) % len(src_idx)]
            noise = rng.normal(0, noise_std, size=(needed, X.shape[1])).astype(np.float32)
            X_synth = X[rep_idx] + noise * feat_std
            y_synth = np.full(needed, cls, dtype=np.int32)
            X_aug.append(X_synth)
            y_aug.append(y_synth)

    return np.vstack(X_aug), np.concatenate(y_aug)


def build_sample_weights(y_remapped):
    ulabels, counts = np.unique(y_remapped, return_counts=True)
    n_cls = len(ulabels)

    raw_weights = {c: len(y_remapped) / (n_cls * cnt) for c, cnt in zip(ulabels, counts)}
    min_w = min(raw_weights.values())
    weights = {c: min(w / min_w, MAX_CLASS_WEIGHT) for c, w in raw_weights.items()}

    sw = np.array([weights[y] for y in y_remapped], dtype=np.float32)
    return sw, weights


# ============================================================================
# ONE LOEO FOLD
# ============================================================================

def run_one_fold(train_events, test_event, farm_id, verbose=True):
    fault_train = [d for d in train_events if d["fault_type"] != "normal"]
    if len(fault_train) < 2:
        return None

    fault_types_in_train = sorted(set(d["fault_type"] for d in fault_train))
    selected_cols, target_groups = get_selected_columns(fault_types_in_train, farm_id)

    if verbose:
        print(f"    Groups: {target_groups}")
        print(f"    Selected raw avg cols: {len(selected_cols)}")

    X_parts = []
    y_parts = []
    feat_cols_ref = None

    for d in fault_train:
        try:
            pred_df, status4_df = load_event_selected(d["filepath"], farm_id, selected_cols=selected_cols)

            # prediction window
            pred_len = len(pred_df)
            if pred_len > 0:
                pred_eng, fc = engineer_features(pred_df, farm_id)
                if feat_cols_ref is None:
                    feat_cols_ref = fc
                use_cols = [c for c in feat_cols_ref if c in pred_eng.columns]
                X_pred = np.nan_to_num(pred_eng[use_cols].values.astype(np.float32),
                                       nan=0.0, posinf=0.0, neginf=0.0)
                y_pred = np.full(len(X_pred), d["label_int"], dtype=np.int32)
                X_parts.append(X_pred)
                y_parts.append(y_pred)

            # capped status-4 rows
            if USE_STATUS4_ROWS and len(status4_df) > 0 and feat_cols_ref is not None:
                max_rows = max(pred_len, 500) * MAX_STATUS4_MULTIPLIER
                if len(status4_df) > max_rows:
                    status4_df = status4_df.sample(max_rows, random_state=42)

                st4_eng, _ = engineer_features(status4_df, farm_id)
                use_cols = [c for c in feat_cols_ref if c in st4_eng.columns]
                X_st4 = np.nan_to_num(st4_eng[use_cols].values.astype(np.float32),
                                      nan=0.0, posinf=0.0, neginf=0.0)
                y_st4 = np.full(len(X_st4), d["label_int"], dtype=np.int32)
                X_parts.append(X_st4)
                y_parts.append(y_st4)

            del pred_df, status4_df
            gc.collect()

        except Exception as e:
            if verbose:
                print(f"    Skip train event {d['event_id']}: {e}")
            continue

    if not X_parts:
        return None

    X_train = np.vstack(X_parts).astype(np.float32)
    y_train = np.concatenate(y_parts).astype(np.int32)
    del X_parts, y_parts
    gc.collect()

    X_train, y_train = oversample_minority_classes(X_train, y_train)

    unique_cls = sorted(np.unique(y_train).tolist())
    n_cls = len(unique_cls)
    remap = {old: new for new, old in enumerate(unique_cls)}
    reverse_remap = {new: old for old, new in remap.items()}

    y_remapped = np.array([remap[y] for y in y_train], dtype=np.int32)
    sample_weights, weights_map = build_sample_weights(y_remapped)

    params = XGB_PARAMS_SMALL.copy()
    if n_cls > 2:
        params["objective"] = "multi:softprob"
        params["eval_metric"] = "mlogloss"
        params["num_class"] = n_cls
    else:
        params["objective"] = "binary:logistic"
        params["eval_metric"] = "logloss"

    clf = xgb.XGBClassifier(**params)

    try:
        clf.fit(X_train, y_remapped, sample_weight=sample_weights)
    except Exception as e:
        if verbose:
            print(f"    Fit failed: {e}")
        return None

    del X_train, y_train, sample_weights
    gc.collect()

    # test event
    try:
        pred_df_test, _ = load_event_selected(test_event["filepath"], farm_id, selected_cols=selected_cols)
        if len(pred_df_test) == 0:
            return None

        pred_eng_test, _ = engineer_features(pred_df_test, farm_id)
        use_cols = [c for c in feat_cols_ref if c in pred_eng_test.columns]

        X_test = np.nan_to_num(pred_eng_test[use_cols].values.astype(np.float32),
                               nan=0.0, posinf=0.0, neginf=0.0)

        if X_test.shape[1] < len(feat_cols_ref):
            pad = np.zeros((len(X_test), len(feat_cols_ref) - X_test.shape[1]), dtype=np.float32)
            X_test = np.hstack([X_test, pad])

        timestamps = pred_eng_test["time_stamp"].values if "time_stamp" in pred_eng_test.columns else None

        probas = clf.predict_proba(X_test)
        preds_remapped = np.argmax(probas, axis=1)
        max_probs = np.max(probas, axis=1)

        preds_conf = np.where(max_probs >= CONFIDENCE_THRESHOLD, preds_remapped, -1)
        preds_orig = np.array([reverse_remap[p] if p >= 0 else -1 for p in preds_conf], dtype=int)

        del pred_df_test, pred_eng_test, X_test
        gc.collect()

    except Exception as e:
        if verbose:
            print(f"    Test failed: {e}")
        return None

    true_label = test_event["label_int"]
    is_fault = test_event["fault_type"] != "normal"

    detected = False
    horizon = None
    detected_window = False

    if is_fault and true_label in unique_cls:
        detected, horizon = detect_failure(preds_orig, true_label, timestamps)
        detected_window = detect_failure_window(preds_orig, true_label, window_frac=0.10)

    valid_preds = preds_orig[preds_orig >= 0]
    if len(valid_preds) > 0:
        pred_int = Counter(valid_preds.tolist()).most_common(1)[0][0]
        int_to_ft = {d["label_int"]: d["fault_type"] for d in fault_train}
        pred_type = int_to_ft.get(pred_int, "unknown")
    else:
        pred_type = "uncertain"

    mean_conf = float(np.mean(max_probs)) if len(max_probs) else 0.0
    uncertain_frac = float(np.mean(preds_orig == -1)) if len(preds_orig) else 1.0

    importance = {}
    if hasattr(clf, "feature_importances_"):
        n_base = min(len(feat_cols_ref), len(clf.feature_importances_))
        for col, imp in zip(feat_cols_ref[:n_base], clf.feature_importances_[:n_base]):
            importance[col] = float(imp)

    status_mark = "✓" if detected else ("~" if detected_window else ("✗" if is_fault else "—"))
    if verbose:
        hor_str = f" ({horizon:.1f}d)" if horizon is not None else ""
        print(f"    E{test_event['event_id']:>4} [{test_event['fault_type']:22}] -> [{pred_type:22}] {status_mark}{hor_str} conf={mean_conf:.2f}")

    return {
        "event_id": test_event["event_id"],
        "farm": farm_id,
        "fault_type": test_event["fault_type"],
        "pred_type": pred_type,
        "label_int": int(true_label),
        "is_fault": bool(is_fault),
        "detected": bool(detected),
        "detected_window": bool(detected_window),
        "horizon_days": round(float(horizon), 2) if horizon is not None else None,
        "mean_confidence": round(mean_conf, 4),
        "uncertain_frac": round(uncertain_frac, 4),
        "n_train_events": len(fault_train),
        "n_classes_train": n_cls,
        "n_selected_cols": len(selected_cols),
        "sensor_groups": target_groups,
        "feature_importance": dict(sorted(importance.items(), key=lambda x: x[1], reverse=True)[:12]),
    }


# ============================================================================
# RUN LOEO PER FARM
# ============================================================================

def run_farm_loeo(farm_id, dataset):
    farm_events = [d for d in dataset if d["farm"] == farm_id]
    if not farm_events:
        return []

    fault_types = sorted(set(d["fault_type"] for d in farm_events))
    label_map = {ft: i for i, ft in enumerate(fault_types)}

    for d in farm_events:
        d["label_int"] = label_map[d["fault_type"]]

    anomaly_events = [d for d in farm_events if d["label"] == "anomaly"]
    normal_events  = [d for d in farm_events if d["label"] == "normal"]

    print(f"Farm {farm_id}:")
    print(f"  anomaly events: {len(anomaly_events)}")
    print(f"  normal events : {len(normal_events)}")
    print(f"  fault types   : {fault_types}\n")

    results = []

    for i, test_event in enumerate(farm_events, 1):
        print(f"  [{i}/{len(farm_events)}] LOEO test event E{test_event['event_id']} [{test_event['fault_type']}]")
        train_events = [d for d in farm_events if d["event_id"] != test_event["event_id"]]

        result = run_one_fold(train_events, test_event, farm_id, verbose=True)
        if result is not None:
            results.append(result)
        print()

    return results


# ============================================================================
# METRICS
# ============================================================================

def compute_metrics(results, farm_id):
    fault_results = [r for r in results if r["is_fault"] and r["fault_type"] != "normal"]
    if not fault_results:
        return {}

    total = len(fault_results)
    detected = sum(1 for r in fault_results if r["detected"])
    detected_window = sum(1 for r in fault_results if r["detected_window"])
    rate = detected / total if total > 0 else 0.0
    rate_window = detected_window / total if total > 0 else 0.0

    classifiable = [r for r in fault_results if r["pred_type"] not in ("uncertain", "unknown", None)]
    n_correct = sum(1 for r in classifiable if r["pred_type"] == r["fault_type"])
    class_acc = n_correct / len(classifiable) if classifiable else 0.0

    uncertain_n = sum(1 for r in fault_results if r["pred_type"] in ("uncertain", None))
    mean_conf = float(np.mean([r["mean_confidence"] for r in fault_results])) if fault_results else 0.0
    avg_horizon = float(np.mean([r["horizon_days"] for r in fault_results if r["horizon_days"] is not None])) if any(r["horizon_days"] is not None for r in fault_results) else 0.0

    by_type = {}
    for r in fault_results:
        ft = r["fault_type"]
        if ft not in by_type:
            by_type[ft] = {
                "total": 0,
                "detected": 0,
                "detected_window": 0,
                "correct": 0,
                "classifiable": 0,
                "horizons": [],
            }
        by_type[ft]["total"] += 1
        if r["detected"]:
            by_type[ft]["detected"] += 1
        if r["detected_window"]:
            by_type[ft]["detected_window"] += 1
        if r["pred_type"] not in ("uncertain", "unknown", None):
            by_type[ft]["classifiable"] += 1
            if r["pred_type"] == ft:
                by_type[ft]["correct"] += 1
        if r["horizon_days"] is not None:
            by_type[ft]["horizons"].append(r["horizon_days"])

    print("-" * 80)
    print(f"Farm {farm_id} — classification results")
    print("-" * 80)
    print(f"Detection rate (5-consecutive): {detected}/{total} = {rate:.1%}")
    print(f"Detection rate (last-10% win): {detected_window}/{total} = {rate_window:.1%}")
    print(f"Classification accuracy      : {n_correct}/{len(classifiable)} = {class_acc:.1%}")
    print(f"Uncertain predictions        : {uncertain_n}")
    print(f"Mean confidence              : {mean_conf:.2f}")
    print(f"Average horizon              : {avg_horizon:.1f} days\n")

    print(f"{'Fault type':25} {'Det':>4} {'Win':>4} {'Tot':>4} {'Det%':>6} {'Win%':>6} {'Cls%':>6} {'Horizon':>10}")
    print("-" * 80)
    for ft in sorted(by_type.keys()):
        v = by_type[ft]
        det_rate = v["detected"] / v["total"] if v["total"] else 0.0
        win_rate = v["detected_window"] / v["total"] if v["total"] else 0.0
        cls_rate = v["correct"] / v["classifiable"] if v["classifiable"] else 0.0
        hor = float(np.mean(v["horizons"])) if v["horizons"] else 0.0
        print(f"{ft:25} {v['detected']:4d} {v['detected_window']:4d} {v['total']:4d} {det_rate:6.1%} {win_rate:6.1%} {cls_rate:6.1%} {hor:8.1f}d")
    print()

    # json-safe by_type copy
    by_type_clean = {}
    for ft, v in by_type.items():
        by_type_clean[ft] = {
            "total": int(v["total"]),
            "detected": int(v["detected"]),
            "detected_window": int(v["detected_window"]),
            "correct": int(v["correct"]),
            "classifiable": int(v["classifiable"]),
            "avg_horizon_days": round(float(np.mean(v["horizons"])), 2) if v["horizons"] else 0.0,
        }

    return {
        "farm": farm_id,
        "total_faults": int(total),
        "detected": int(detected),
        "detected_window": int(detected_window),
        "detection_rate": round(float(rate), 4),
        "detection_rate_window": round(float(rate_window), 4),
        "classification_acc": round(float(class_acc), 4),
        "n_uncertain": int(uncertain_n),
        "mean_confidence": round(float(mean_conf), 4),
        "avg_horizon_days": round(float(avg_horizon), 2),
        "by_type": by_type_clean,
    }


# ============================================================================
# PLOTS
# ============================================================================

def plot_detection_rate(all_metrics):
    farms = [f for f in ["A", "B", "C"] if f in all_metrics]
    if not farms:
        return

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(farms))
    w = 0.35

    det = [all_metrics[f]["detection_rate"] for f in farms]
    cls = [all_metrics[f]["classification_acc"] for f in farms]

    b1 = ax.bar(x - w/2, det, w, label="Detection rate", alpha=0.85)
    b2 = ax.bar(x + w/2, cls, w, label="Classification accuracy", alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([f"Farm {f}" for f in farms])
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Rate")
    ax.set_title("Stage 2 — detection vs classification accuracy", fontweight="bold")
    ax.bar_label(b1, fmt="%.2f", padding=2)
    ax.bar_label(b2, fmt="%.2f", padding=2)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "01_detection_vs_classification.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_horizon(all_metrics):
    farms = [f for f in ["A", "B", "C"] if f in all_metrics]
    if not farms:
        return

    fig, ax = plt.subplots(figsize=(8, 5))
    vals = [all_metrics[f]["avg_horizon_days"] for f in farms]
    bars = ax.bar([f"Farm {f}" for f in farms], vals, alpha=0.85)
    ax.set_ylabel("Average detection horizon (days)")
    ax.set_title("Stage 2 — average early warning horizon", fontweight="bold")
    ax.bar_label(bars, fmt="%.1f", padding=2)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "02_horizon_days.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_farm_c_by_type(all_results):
    farm_c = [r for r in all_results if r["farm"] == "C" and r["is_fault"]]
    if not farm_c:
        return

    by_type = {}
    for r in farm_c:
        ft = r["fault_type"]
        if ft not in by_type:
            by_type[ft] = {"total": 0, "detected": 0}
        by_type[ft]["total"] += 1
        if r["detected"]:
            by_type[ft]["detected"] += 1

    types = sorted(by_type.keys())
    rates = [by_type[t]["detected"] / by_type[t]["total"] for t in types]

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(types, rates, alpha=0.85)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Detection rate")
    ax.set_title("Farm C — detection rate by fault type", fontweight="bold")
    ax.bar_label(bars, fmt="%.2f", padding=2)
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "03_farmC_by_type.png"), dpi=150, bbox_inches="tight")
    plt.close()


# ============================================================================
# MAIN
# ============================================================================

def json_safe(obj):
    if isinstance(obj, dict):
        return {k: json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_safe(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return str(obj)
    if pd.isna(obj) if not isinstance(obj, (list, dict, tuple, np.ndarray)) else False:
        return None
    return obj


def main():
    print("Running LOEO classification...\n")
    start = datetime.now()

    all_results = []
    all_metrics = {}

    for farm in ["A", "B", "C"]:
        print("=" * 80)
        print(f"FARM {farm} — LOEO CLASSIFICATION")
        print("=" * 80)

        farm_results = run_farm_loeo(farm, dataset)
        all_results.extend(farm_results)

        if farm_results:
            metrics = compute_metrics(farm_results, farm)
            if metrics:
                all_metrics[farm] = metrics

    print("Generating plots...\n")
    plot_detection_rate(all_metrics)
    plot_horizon(all_metrics)
    plot_farm_c_by_type(all_results)

    results_json = {
        "timestamp": datetime.now().isoformat(),
        "script": "script_2_fault_classification_clean",
        "base_path": BASE_PATH,
        "parameters": {
            "confidence_threshold": CONFIDENCE_THRESHOLD,
            "consecutive_window": CONSECUTIVE_WINDOW,
            "consecutive_window_short": CONSECUTIVE_WINDOW_SHORT,
            "short_window_threshold": SHORT_WINDOW_THRESHOLD,
            "min_examples_for_class": MIN_EXAMPLES_FOR_CLASS,
            "oversample_noise_std": OVERSAMPLE_NOISE_STD,
            "max_class_weight": MAX_CLASS_WEIGHT,
            "use_status4_rows": USE_STATUS4_ROWS,
            "max_status4_multiplier": MAX_STATUS4_MULTIPLIER,
        },
        "metrics": all_metrics,
        "results": all_results,
    }

    json_path = os.path.join(OUTPUT_DIR, "results_stage2_clean.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(json_safe(results_json), f, indent=2)

    csv_path = os.path.join(OUTPUT_DIR, "event_results_stage2_clean.csv")
    pd.DataFrame(all_results).drop(columns=["feature_importance"], errors="ignore").to_csv(csv_path, index=False)

    metrics_path = os.path.join(REPORT_DIR, "classification_summary_clean.csv")
    pd.DataFrame([{"farm": f, **all_metrics[f]} for f in all_metrics]).to_csv(metrics_path, index=False)

    print("\nSaved outputs:")
    print(f"  {json_path}")
    print(f"  {csv_path}")
    print(f"  {metrics_path}")

    print("\n" + "=" * 80)
    print("SCRIPT 2 COMPLETE")
    print("=" * 80)
    for farm in ["A", "B", "C"]:
        if farm in all_metrics:
            m = all_metrics[farm]
            print(
                f"Farm {farm}: "
                f"det={m['detected']}/{m['total_faults']} ({m['detection_rate']:.3f}) | "
                f"win={m['detected_window']}/{m['total_faults']} ({m['detection_rate_window']:.3f}) | "
                f"cls={m['classification_acc']:.3f} | "
                f"horizon={m['avg_horizon_days']:.1f}d"
            )

    print(f"\nRuntime: {datetime.now() - start}")


if __name__ == "__main__":
    main()

SCRIPT 2 — CARE FAULT CLASSIFICATION (CLEAN REBUILD)
Base directory: E:\project\CARE_To_Compare

Farm A registry:
  Column mappings: 81
  other                  sensors= 15  cols= 18  model= 16
  electrical             sensors= 14  cols= 29  model= 29
  generator_other        sensors=  7  cols= 10  model= 10
  generator_stator       sensors=  3  cols=  3  model=  3
  transformer            sensors=  3  cols=  3  model=  3
  rotor                  sensors=  2  cols=  5  model=  4
  environment            sensors=  2  cols=  2  model=  2
  generator_bearing      sensors=  2  cols=  2  model=  2
  pitch                  sensors=  1  cols=  4  model=  0
  cooling_water          sensors=  1  cols=  1  model=  1
  gearbox_bearing        sensors=  1  cols=  1  model=  1
  gearbox_other          sensors=  1  cols=  1  model=  1

Farm B registry:
  Column mappings: 252
  electrical             sensors= 20  cols= 80  model= 64
  transformer            sensors=  9  cols= 36  model= 36
  pitch    

In [22]:
"""
================================================================================
SCRIPT 3 — CROSS-FARM FAULT CLASSIFICATION VALIDATION (CLEAN REBUILD)
================================================================================
Tests whether fault signatures learned on one wind farm generalise to another.

Design:
  - Train on ALL events from a source farm
  - Predict ALL events in a target farm
  - Use farm-agnostic subsystem aggregate features
  - Use per-farm z-score normalisation based on normal events only
  - Evaluate transfer directions:
      A -> C
      C -> A
      C -> B
      B -> A
      B -> C

Representation:
  Raw SCADA channels cannot transfer directly because farms have different
  sensor counts and names. Instead, each event is represented by subsystem-
  level aggregate features computed at timestep level:
      mean, std, min, max, trend, roll1h_mean, roll1h_std, roll3h_mean, roll3h_std
  across shared subsystem groups.

Outputs:
  - cross_farm_outputs/results_cross_farm.json
  - cross_farm_outputs/cross_farm_summary.csv
  - cross_farm_outputs/plots/*.png

Author: Peter
Dataset: CARE benchmark (Gück et al., 2024)
"""

# ============================================================================
# IMPORTS
# ============================================================================

import os
import gc
import re
import json
import warnings
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import xgboost as xgb

warnings.filterwarnings("ignore")

# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_PATH = r"E:\project\CARE_To_Compare"

FARMS = {
    "A": os.path.join(BASE_PATH, "Wind Farm A"),
    "B": os.path.join(BASE_PATH, "Wind Farm B"),
    "C": os.path.join(BASE_PATH, "Wind Farm C"),
}

DATASET_DIRS = {
    "A": os.path.join(FARMS["A"], "datasets"),
    "B": os.path.join(FARMS["B"], "datasets"),
    "C": os.path.join(FARMS["C"], "datasets"),
}

FEATURE_DESC = {
    "A": os.path.join(FARMS["A"], "feature_description.csv"),
    "B": os.path.join(FARMS["B"], "feature_description.csv"),
    "C": os.path.join(FARMS["C"], "feature_description.csv"),
}

EVENT_INFO_PATH = os.path.join(BASE_PATH, "df_all__1_.csv")

OUTPUT_DIR = os.path.join(BASE_PATH, "cross_farm_outputs")
PLOT_DIR   = os.path.join(OUTPUT_DIR, "plots")
REPORT_DIR = os.path.join(OUTPUT_DIR, "reports")

for d in [OUTPUT_DIR, PLOT_DIR, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)

CSV_SEP       = ";"
FARM_NAME_MAP = {"Farm A": "A", "Farm B": "B", "Farm C": "C"}
NORMAL_STATUS = {0, 2}

CONSECUTIVE_WINDOW = 5
CONFIDENCE_THRESHOLD = 0.40
MAX_CLASS_WEIGHT = 8.0

XGB_PARAMS = dict(
    max_depth=4,
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.80,
    colsample_bylevel=0.80,
    reg_lambda=2.5,
    reg_alpha=0.15,
    min_child_weight=2,
    gamma=0.1,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

META_COLS = {"time_stamp", "asset_id", "id", "train_test", "status_type_id"}

# Only fault types that make physical sense to compare across farms
COMPARABLE_FAULT_TYPES = {
    "transformer",
    "hydraulic",
    "gearbox",
    "gearbox_oil",
    "generator_bearing",
    "rotor_bearing",
}

# Shared subsystem groups used in transfer representation
CROSS_FARM_GROUPS = [
    "gearbox_bearing",
    "gearbox_oil",
    "gearbox_other",
    "generator_bearing",
    "generator_stator",
    "generator_cooling",
    "generator_other",
    "hydraulic",
    "rotor_bearing",
    "transformer",
    "electrical",
    "wind",
    "environment",
    "control",
    "pitch",
    "vibration",
]

ROLL_WINDOWS = [(6, "1h"), (18, "3h")]

# ============================================================================
# FAULT LABELS
# ============================================================================

FAULT_LABELS = {
    "A": {
        68: "transformer",
        22: "hydraulic",
        72: "gearbox",
        73: "hydraulic",
        0:  "generator_bearing",
        26: "hydraulic",
        40: "generator_bearing",
        42: "hydraulic",
        10: "gearbox",
        45: "hydraulic",
        84: "hydraulic",
        51: "gearbox",
    },
    "B": {
        34: "transformer",
        7:  "transformer",
        53: "rotor_bearing",
        27: "rotor_bearing",
        19: "transformer",
        77: "rotor_bearing",
    },
    "C": {
        55: "pitch_electrical",
        81: "converter",
        47: "hydraulic",
        12: "gearbox_oil",
        4:  "pitch_electrical",
        18: "rotor_brake",
        28: "pitch_electrical",
        39: "gearbox_oil",
        66: "pitch_electrical",
        15: "pitch_electrical",
        78: "rotor_brake",
        79: "communication",
        30: "pitch_electrical",
        33: "pitch_mechanical",
        11: "pitch_electrical",
        44: "cooling_water",
        49: "gearbox_oil",
        31: "communication",
        67: "transformer",
        9:  "yaw",
        91: "pitch_electrical",
        5:  "electrical_internal",
        90: "communication",
        70: "pitch_electrical",
        35: "converter",
        16: "pitch_electrical",
        76: "pitch_electrical",
    },
}

# ============================================================================
# SUBSYSTEM RULES
# ============================================================================

SUBSYSTEM_RULES = [
    (10,  "gearbox_bearing",  ["planetary bearing", "gearbox bearing",
                               "bearing on high speed shaft"]),
    (11,  "gearbox_oil",      ["gearbox oil", "gear oil", "oil inlet temperature",
                               "oil pressure", "oil level", "oil tank",
                               "oil temperature"]),
    (12,  "gearbox_other",    ["gearbox rotational", "gearbox speed",
                               "gearbox main shaft", "gearbox"]),
    (20,  "generator_bearing",["generator bearing"]),
    (21,  "generator_stator", ["stator winding", "stator"]),
    (22,  "generator_cooling",["generator cooling", "cooling air",
                               "cooling water temp. generator",
                               "generator temperature", "cooling air inlet"]),
    (23,  "generator_other",  ["generator acceleration", "generator angle",
                               "generator rpm", "generator converter",
                               "generator rms", "generator"]),
    (30,  "rotor_bearing",    ["rotor bearing", "axial bearing",
                               "rotor bearing inner ring"]),
    (40,  "hydraulic",        ["hydraulic pressure", "hydraulic oil",
                               "hydraulic pump", "hydraulic tank",
                               "hydraulic group", "aggregate system pressure",
                               "azimuth circuit", "azimuth memory",
                               "rotor brake", "locking memory",
                               "pollution indicator", "oil container"]),
    (50,  "pitch",            ["pitch angle", "pitch motor", "pitch axis",
                               "position motor axis", "position rotor blade",
                               "motor temperature kty", "motor temperature pt",
                               "axis 1 motor", "axis 2 motor", "axis 3 motor",
                               "axis 1 voltage", "axis 2 voltage", "axis 3 voltage",
                               "battery charge", "battery discharge",
                               "rpm motor axis", "min pitch",
                               "motor current axis", "motor temperature axis"]),
    (60,  "vibration",        ["vibration", "drive train vibration",
                               "tower vibration"]),
    (70,  "transformer",      ["transformer", "transformator",
                               "hv transformer", "transformer cell"]),
    (80,  "electrical",       ["voltage", "current", "frequency",
                               "active power", "reactive power",
                               "apparent power", "grid power", "grid voltage",
                               "grid frequency", "available power",
                               "inverter torque", "cable load",
                               "dc link voltage", "internal consumption"]),
    (90,  "wind",             ["wind speed", "wind direction", "anemometer",
                               "estimated windspeed", "absolute wind",
                               "relative wind"]),
    (100, "environment",      ["ambient temperature", "outside temperature",
                               "nacelle temperature", "nacelle external",
                               "hub temperature", "board temperature",
                               "electrical cabinet", "converter room",
                               "control box temperature", "nacelle outside",
                               "platform temperature"]),
    (110, "cooling_water",    ["water temperature", "water pressure",
                               "water flow", "water conductivity",
                               "water cooler", "cooling water",
                               "flow meter stator"]),
    (120, "control",          ["yaw", "nacelle direction",
                               "estimated tower", "compensation angle",
                               "rotor speed", "rotor rpm", "bypass valve",
                               "aeration", "pressure difference",
                               "rotational speed aerator", "cable load"]),
    (130, "temperature",      ["temperature", "temp."]),
]

# ============================================================================
# HELPERS
# ============================================================================

STAT_MAP = {"average": "avg", "std_dev": "std", "maximum": "max", "minimum": "min"}

def _clean_desc(text):
    if pd.isna(text) or str(text).strip() in ("", "nan"):
        return "unknown"
    text = str(text)
    for old, new in [("���C", "degC"), ("°C", "degC"), ("ºC", "degC"),
                     ("½C", "degC"), ("Celsius", "degC"),
                     ("���", "deg"), ("°", "deg"), ("º", "deg"), ("½", "")]:
        text = text.replace(old, new)
    text = re.sub(r"[^\x00-\x7F]+", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", "_", text.strip())
    return re.sub(r"_+", "_", text).strip("_")

def _clean_unit(unit):
    if pd.isna(unit) or str(unit).strip().lower() in ("", "nan", "-", "none"):
        return ""
    unit = str(unit)
    for old, new in [("���C", "degC"), ("°C", "degC"), ("ºC", "degC"),
                     ("½C", "degC"), ("Celsius", "degC"),
                     ("°", "deg"), ("º", "deg"), ("½", ""),
                     ("/", "_per_"), ("%", "pct"), (" ", ""), ("^", ""), (".", "")]:
        unit = unit.replace(old, new)
    unit = re.sub(r"[^\x00-\x7F]+", "", unit)
    return re.sub(r"[^\w]", "", unit)

def _assign_group(description, is_counter=False, is_angle=False):
    if is_counter:
        return "counter"
    if is_angle:
        return "angle"
    desc_lower = description.lower()
    for _, group, keywords in sorted(SUBSYSTEM_RULES, key=lambda x: x[0]):
        if any(kw in desc_lower for kw in keywords):
            return group
    return "other"

def json_safe(obj):
    if isinstance(obj, dict):
        return {k: json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_safe(v) for v in obj]
    if isinstance(obj, set):
        return sorted(list(obj))
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return str(obj)
    if pd.isna(obj) if not isinstance(obj, (list, dict, tuple, set, np.ndarray)) else False:
        return None
    return obj

# ============================================================================
# SENSOR REGISTRY
# ============================================================================

def build_sensor_registry(farm_id):
    desc_path = FEATURE_DESC[farm_id]
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(desc_path, sep=";", encoding=enc)
            break
        except UnicodeDecodeError:
            continue

    rename_dict, group_dict, rows = {}, {}, []
    seen = {}

    for _, row in df.iterrows():
        sid        = str(row["sensor_name"]).strip()
        desc_raw   = str(row.get("description", "")).strip()
        unit_raw   = str(row.get("unit", "")).strip()
        is_counter = bool(row.get("is_counter", False))
        is_angle   = bool(row.get("is_angle", False))
        group      = _assign_group(desc_raw, is_counter, is_angle)

        desc_clean = _clean_desc(desc_raw)
        unit_clean = _clean_unit(unit_raw)

        stats_raw = str(row.get("statistics_type", "")).strip()
        stats = [] if stats_raw.lower() in ("nan", "") else \
                [s.strip() for s in stats_raw.split(",") if s.strip()]

        entries = []
        if stats:
            for sr in stats:
                sc = STAT_MAP.get(sr, sr)
                entries.append((f"{sid}_{sc}", sc))
        else:
            entries.append((sid, ""))

        for old_col, stat_clean in entries:
            parts = [desc_clean] + ([unit_clean] if unit_clean else [])
            if stat_clean:
                parts.append(stat_clean)
            new_col = "_".join(parts)

            if new_col in seen and seen[new_col] != old_col:
                new_col = f"{new_col}_{sid}"
            seen[new_col] = old_col

            rename_dict[old_col] = new_col
            group_dict[new_col]  = group

            rows.append({
                "farm": farm_id,
                "sensor_id": sid,
                "old_col": old_col,
                "new_col": new_col,
                "description": desc_raw,
                "unit": unit_raw,
                "stat": stat_clean,
                "group": group,
                "is_counter": is_counter,
                "is_angle": is_angle,
                "use_in_model": not (is_counter or is_angle),
            })

    return rename_dict, group_dict, pd.DataFrame(rows)

print("=" * 80)
print("SCRIPT 3 — CROSS-FARM VALIDATION (CLEAN REBUILD)")
print("=" * 80)
print(f"Base directory: {BASE_PATH}\n")

rename_dicts = {}
group_dicts  = {}
registries   = {}

for farm in ["A", "B", "C"]:
    rd, gd, reg = build_sensor_registry(farm)
    rename_dicts[farm] = rd
    group_dicts[farm]  = gd
    registries[farm]   = reg

    print(f"Farm {farm} registry:")
    print(f"  Column mappings: {len(rd)}")

    summary = (
        reg[reg["stat"].isin(["avg", ""])]
        .groupby("group")
        .agg(
            sensors=("sensor_id", "nunique"),
            cols=("new_col", "count"),
            usable=("use_in_model", "sum"),
        )
        .sort_values(["sensors", "cols"], ascending=False)
    )

    for group, row in summary.head(16).iterrows():
        print(f"  {group:22} sensors={int(row['sensors']):3d}  cols={int(row['cols']):3d}  usable={int(row['usable']):3d}")
    print()

all_registry = pd.concat(registries.values(), ignore_index=True)
all_registry.to_csv(os.path.join(REPORT_DIR, "sensor_registry_cross_farm.csv"), index=False)

# ============================================================================
# DATASET REGISTRY
# ============================================================================

event_info = pd.read_csv(EVENT_INFO_PATH)
event_info["farm"]        = event_info["Farm"].map(FARM_NAME_MAP)
event_info["event_id"]    = event_info["event_id"].astype(str)
event_info["event_label"] = event_info["event_label"].astype(str).str.strip()

dataset = []
for _, row in event_info.iterrows():
    farm = row["farm"]
    if pd.isna(farm):
        continue
    eid = row["event_id"]
    filepath = os.path.join(DATASET_DIRS[farm], f"{eid}.csv")
    if not os.path.exists(filepath):
        continue

    eid_int = int(eid) if eid.isdigit() else -1
    fault_type = FAULT_LABELS.get(farm, {}).get(eid_int, "normal")
    if row["event_label"] == "normal":
        fault_type = "normal"

    dataset.append({
        "event_id": eid,
        "farm": farm,
        "label": row["event_label"],
        "fault_type": fault_type,
        "filepath": filepath,
    })

print("Event counts:")
for farm in ["A", "B", "C"]:
    subset = [d for d in dataset if d["farm"] == farm]
    counts = Counter(d["fault_type"] for d in subset if d["label"] == "anomaly")
    print(f"  Farm {farm}: total={len(subset)}  anomaly={sum(d['label']=='anomaly' for d in subset)}  normal={sum(d['label']=='normal' for d in subset)}")
    print(f"    Fault types: {dict(counts)}")
print()

# ============================================================================
# DATA LOADING
# ============================================================================

def get_excluded_cols(farm_id):
    reg = registries[farm_id]
    return set(reg.loc[~reg["use_in_model"], "new_col"])

def handle_zero_runs(df, farm_id, min_run=6):
    excluded = get_excluded_cols(farm_id)
    numeric_cols = [c for c in df.columns if c not in META_COLS and pd.api.types.is_numeric_dtype(df[c])]
    for col in numeric_cols:
        if col in excluded:
            continue
        vals = df[col].values.copy().astype(np.float32)
        n, i = len(vals), 0
        while i < n:
            if vals[i] == 0.0:
                start = i
                while i < n and vals[i] == 0.0:
                    i += 1
                if i - start >= min_run:
                    vals[start:i] = np.nan
            else:
                i += 1
        df[col] = vals
    return df

def load_event(filepath, farm_id, chunk_size=10000):
    rename_dict = rename_dicts[farm_id]
    chunks = []

    for chunk in pd.read_csv(filepath, sep=CSV_SEP, chunksize=chunk_size,
                             low_memory=False, encoding="latin-1"):
        chunk = chunk.rename(columns=rename_dict)
        chunk = handle_zero_runs(chunk, farm_id)
        num_cols = [c for c in chunk.columns if c not in META_COLS and pd.api.types.is_numeric_dtype(chunk[c])]
        chunk[num_cols] = chunk[num_cols].astype(np.float32)
        chunks.append(chunk)

    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()

    train_df = df[df["train_test"] == "train"].copy()
    pred_df  = df[df["train_test"] == "prediction"].copy()

    if "status_type_id" in train_df.columns:
        train_df = train_df[train_df["status_type_id"].isin(NORMAL_STATUS)]

    del df
    gc.collect()
    return train_df, pred_df

# ============================================================================
# CROSS-FARM SUBSYSTEM FEATURES
# ============================================================================

def compute_group_feature_names():
    names = []
    for g in CROSS_FARM_GROUPS:
        for stat in ["mean", "std", "min", "max", "trend", "roll1h_mean", "roll1h_std", "roll3h_mean", "roll3h_std"]:
            names.append(f"{g}_{stat}")
    return names

FEATURE_NAMES = compute_group_feature_names()

def compute_cross_farm_features(df, farm_id):
    """
    Returns a timestep-level feature DataFrame with fixed schema across farms.
    """
    gdict = group_dicts[farm_id]
    excluded = get_excluded_cols(farm_id)

    avg_cols = [
        c for c in df.columns
        if c not in META_COLS
        and c not in excluded
        and "avg" in c.lower()
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    group_to_cols = defaultdict(list)
    for col in avg_cols:
        group = gdict.get(col, "other")
        if group in CROSS_FARM_GROUPS:
            group_to_cols[group].append(col)

    out = {}
    n = len(df)

    for group in CROSS_FARM_GROUPS:
        cols = group_to_cols.get(group, [])
        if cols:
            mat = np.nan_to_num(df[cols].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            group_mean = np.mean(mat, axis=1).astype(np.float32)
            group_std  = np.std(mat, axis=1).astype(np.float32)
            group_min  = np.min(mat, axis=1).astype(np.float32)
            group_max  = np.max(mat, axis=1).astype(np.float32)

            # trend over 6 timesteps
            group_trend = pd.Series(group_mean).diff(6).fillna(0).values.astype(np.float32)

            out[f"{group}_mean"]  = group_mean
            out[f"{group}_std"]   = group_std
            out[f"{group}_min"]   = group_min
            out[f"{group}_max"]   = group_max
            out[f"{group}_trend"] = group_trend

            for w, wname in ROLL_WINDOWS:
                s = pd.Series(group_mean)
                out[f"{group}_roll{wname}_mean"] = s.rolling(w, min_periods=1).mean().values.astype(np.float32)
                out[f"{group}_roll{wname}_std"]  = s.rolling(w, min_periods=1).std().fillna(0).values.astype(np.float32)
        else:
            # structurally absent group -> zeros
            out[f"{group}_mean"]  = np.zeros(n, dtype=np.float32)
            out[f"{group}_std"]   = np.zeros(n, dtype=np.float32)
            out[f"{group}_min"]   = np.zeros(n, dtype=np.float32)
            out[f"{group}_max"]   = np.zeros(n, dtype=np.float32)
            out[f"{group}_trend"] = np.zeros(n, dtype=np.float32)
            out[f"{group}_roll1h_mean"] = np.zeros(n, dtype=np.float32)
            out[f"{group}_roll1h_std"]  = np.zeros(n, dtype=np.float32)
            out[f"{group}_roll3h_mean"] = np.zeros(n, dtype=np.float32)
            out[f"{group}_roll3h_std"]  = np.zeros(n, dtype=np.float32)

    feat_df = pd.DataFrame(out, index=df.index)
    feat_df = feat_df[FEATURE_NAMES].fillna(0).astype(np.float32)
    return feat_df

# ============================================================================
# BASELINES FOR NORMALISATION
# ============================================================================

def compute_farm_baseline(farm_id, dataset):
    """
    Baseline is computed from normal-event training windows only.
    """
    farm_normals = [d for d in dataset if d["farm"] == farm_id and d["label"] == "normal"]
    rows = []

    print(f"Computing baseline for Farm {farm_id} from {len(farm_normals)} normal events...")

    for d in farm_normals:
        try:
            train_df, _ = load_event(d["filepath"], farm_id)
            if len(train_df) == 0:
                continue
            feat_df = compute_cross_farm_features(train_df, farm_id)
            rows.append(feat_df.values.astype(np.float32))
            del train_df, feat_df
            gc.collect()
        except Exception:
            continue

    if not rows:
        mu = np.zeros(len(FEATURE_NAMES), dtype=np.float32)
        sd = np.ones(len(FEATURE_NAMES), dtype=np.float32)
        return mu, sd

    X = np.vstack(rows).astype(np.float32)
    mu = np.mean(X, axis=0).astype(np.float32)
    sd = np.std(X, axis=0).astype(np.float32)
    sd = np.where(sd < 1e-6, 1.0, sd).astype(np.float32)

    print(f"  Baseline matrix: rows={X.shape[0]:,}  features={X.shape[1]}")
    del X, rows
    gc.collect()
    return mu, sd

def zscore_with_baseline(X, mu, sd):
    return ((X - mu) / sd).astype(np.float32)

# ============================================================================
# SOURCE MATRIX BUILD
# ============================================================================

def build_source_training_matrix(source_farm):
    """
    Uses:
      - normal events -> training window
      - anomaly events -> prediction window
    """
    source_events = [d for d in dataset if d["farm"] == source_farm]
    fault_types = sorted(set(d["fault_type"] for d in source_events))
    label_map = {ft: i for i, ft in enumerate(fault_types)}
    reverse_map = {i: ft for ft, i in label_map.items()}

    X_parts = []
    y_parts = []

    print(f"\nBuilding source training matrix for Farm {source_farm}...")

    for d in source_events:
        try:
            train_df, pred_df = load_event(d["filepath"], source_farm)

            if d["fault_type"] == "normal":
                source_df = train_df
            else:
                source_df = pred_df

            if len(source_df) == 0:
                continue

            feat_df = compute_cross_farm_features(source_df, source_farm)
            X = feat_df.values.astype(np.float32)
            y = np.full(len(X), label_map[d["fault_type"]], dtype=np.int32)

            X_parts.append(X)
            y_parts.append(y)

            del train_df, pred_df, source_df, feat_df, X, y
            gc.collect()

        except Exception as e:
            print(f"  Skip source event {d['event_id']}: {e}")

    if not X_parts:
        return None, None, None, None

    X_train = np.vstack(X_parts).astype(np.float32)
    y_train = np.concatenate(y_parts).astype(np.int32)

    print(f"  Source matrix: rows={X_train.shape[0]:,}  features={X_train.shape[1]}  classes={len(label_map)}")
    return X_train, y_train, label_map, reverse_map

# ============================================================================
# MODEL TRAINING
# ============================================================================

def train_cross_farm_model(X_train, y_train):
    unique_cls = sorted(np.unique(y_train).tolist())
    remap = {old: new for new, old in enumerate(unique_cls)}
    reverse_remap = {new: old for old, new in remap.items()}
    y_r = np.array([remap[y] for y in y_train], dtype=np.int32)

    ulabels, counts = np.unique(y_r, return_counts=True)
    raw_weights = {c: len(y_r) / (len(unique_cls) * cnt) for c, cnt in zip(ulabels, counts)}
    min_w = min(raw_weights.values())
    weights = {c: min(raw_weights[c] / min_w, MAX_CLASS_WEIGHT) for c in raw_weights}
    sample_weight = np.array([weights[y] for y in y_r], dtype=np.float32)

    params = XGB_PARAMS.copy()
    if len(unique_cls) > 2:
        params["objective"] = "multi:softprob"
        params["eval_metric"] = "mlogloss"
        params["num_class"] = len(unique_cls)
    else:
        params["objective"] = "binary:logistic"
        params["eval_metric"] = "logloss"

    clf = xgb.XGBClassifier(**params)
    clf.fit(X_train, y_r, sample_weight=sample_weight)

    return clf, remap, reverse_remap

# ============================================================================
# DETECTION RULE
# ============================================================================

def detect_failure(predictions, true_label, timestamps):
    n = len(predictions)
    w = CONSECUTIVE_WINDOW

    for i in range(n - w + 1):
        if np.all(predictions[i:i+w] == true_label):
            if timestamps is not None and len(timestamps) > 0:
                det_ts = pd.Timestamp(timestamps[i])
                end_ts = pd.Timestamp(timestamps[-1])
                horizon_days = (end_ts - det_ts).total_seconds() / 86400
            else:
                horizon_days = (n - i) * (10 / 60 / 24)
            return True, horizon_days

    return False, None

# ============================================================================
# TARGET EVALUATION
# ============================================================================

def evaluate_transfer(source_farm, target_farm, clf, remap, reverse_remap,
                      source_label_map, source_reverse_map,
                      target_mu, target_sd):
    target_events = [d for d in dataset if d["farm"] == target_farm]
    results = []

    print(f"\nEvaluating {source_farm} -> {target_farm} on {len(target_events)} events...")

    for d in target_events:
        try:
            _, pred_df = load_event(d["filepath"], target_farm)
            if len(pred_df) == 0:
                continue

            feat_df = compute_cross_farm_features(pred_df, target_farm)
            X_test = feat_df.values.astype(np.float32)
            X_test = zscore_with_baseline(X_test, target_mu, target_sd)

            probas = clf.predict_proba(X_test)
            preds_r = np.argmax(probas, axis=1)
            max_probs = np.max(probas, axis=1)
            preds_r = np.where(max_probs >= CONFIDENCE_THRESHOLD, preds_r, -1)

            preds_source_space = np.array([reverse_remap[p] if p >= 0 else -1 for p in preds_r])
            preds_type = np.array([
                source_reverse_map[p] if p >= 0 and p in source_reverse_map else "uncertain"
                for p in preds_source_space
            ])

            timestamps = pred_df["time_stamp"].values if "time_stamp" in pred_df.columns else None

            true_type = d["fault_type"]
            true_label = source_label_map.get(true_type, -1)

            detected = False
            horizon = None
            if d["label"] == "anomaly" and true_label >= 0:
                detected, horizon = detect_failure(preds_source_space, true_label, timestamps)

            valid_preds = [p for p in preds_type if p != "uncertain"]
            pred_type_majority = Counter(valid_preds).most_common(1)[0][0] if valid_preds else "uncertain"

            results.append({
                "source_farm": source_farm,
                "target_farm": target_farm,
                "event_id": d["event_id"],
                "fault_type": true_type,
                "pred_type": pred_type_majority,
                "label": d["label"],
                "is_fault": d["label"] == "anomaly",
                "is_comparable": true_type in COMPARABLE_FAULT_TYPES,
                "detected": bool(detected),
                "horizon_days": round(horizon, 2) if horizon is not None else None,
                "mean_confidence": round(float(np.mean(max_probs)), 4),
                "uncertain_frac": round(float(np.mean(preds_r == -1)), 4),
            })

            if d["label"] == "anomaly":
                mark = "✓" if detected else "✗"
                print(f"  E{d['event_id']:>4} [{true_type:20}] -> [{pred_type_majority:20}] {mark}")

            del pred_df, feat_df, X_test, probas, preds_r, preds_source_space, preds_type
            gc.collect()

        except Exception as e:
            print(f"  Target event {d['event_id']} failed: {e}")

    return results

# ============================================================================
# METRICS
# ============================================================================

def compute_transfer_metrics(results, source_farm, target_farm):
    fault_results = [r for r in results if r["is_fault"]]
    comparable = [r for r in fault_results if r["is_comparable"]]

    def pack_metrics(subset):
        if not subset:
            return None
        total = len(subset)
        detected = sum(r["detected"] for r in subset)
        horizons = [r["horizon_days"] for r in subset if r["horizon_days"] is not None]
        by_type = defaultdict(lambda: {"total": 0, "detected": 0, "horizons": []})

        for r in subset:
            ft = r["fault_type"]
            by_type[ft]["total"] += 1
            if r["detected"]:
                by_type[ft]["detected"] += 1
            if r["horizon_days"] is not None:
                by_type[ft]["horizons"].append(r["horizon_days"])

        by_type_out = {}
        for ft, vals in by_type.items():
            by_type_out[ft] = {
                "total": vals["total"],
                "detected": vals["detected"],
                "detection_rate": round(vals["detected"] / vals["total"], 4) if vals["total"] > 0 else 0.0,
                "avg_horizon_days": round(float(np.mean(vals["horizons"])), 2) if vals["horizons"] else None,
            }

        return {
            "total": total,
            "detected": detected,
            "detection_rate": round(detected / total, 4) if total > 0 else 0.0,
            "avg_horizon_days": round(float(np.mean(horizons)), 2) if horizons else None,
            "by_type": by_type_out,
        }

    return {
        "source_farm": source_farm,
        "target_farm": target_farm,
        "all_faults": pack_metrics(fault_results),
        "comparable_faults": pack_metrics(comparable),
    }

# ============================================================================
# PLOTS
# ============================================================================

def plot_transfer_summary(all_metrics):
    directions = []
    comp_rates = []
    all_rates = []
    horizons = []

    for key, m in all_metrics.items():
        directions.append(f"{m['source_farm']}→{m['target_farm']}")
        comp_rates.append((m.get("comparable_faults") or {}).get("detection_rate", 0))
        all_rates.append((m.get("all_faults") or {}).get("detection_rate", 0))
        horizons.append((m.get("comparable_faults") or {}).get("avg_horizon_days", 0) or 0)

    # Detection rates
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(directions))
    w = 0.35

    b1 = ax.bar(x - w/2, comp_rates, w, label="Comparable fault types", alpha=0.85)
    b2 = ax.bar(x + w/2, all_rates,  w, label="All fault types", alpha=0.65)

    ax.set_xticks(x)
    ax.set_xticklabels(directions, fontsize=10)
    ax.set_ylabel("Detection rate", fontsize=10)
    ax.set_ylim(0, 1.1)
    ax.set_title("Cross-farm transfer detection rates", fontsize=11, fontweight="bold")
    ax.bar_label(b1, fmt="%.0f%%", padding=3, fontsize=8)
    ax.bar_label(b2, fmt="%.0f%%", padding=3, fontsize=8)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "01_cross_farm_detection_rates.png"), dpi=150, bbox_inches="tight")
    plt.close()

    # Horizons
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(directions, horizons, alpha=0.85)
    ax.set_ylabel("Average detection horizon (days)", fontsize=10)
    ax.set_title("Cross-farm transfer early-warning horizon", fontsize=11, fontweight="bold")
    ax.bar_label(bars, fmt="%.1f d", padding=3, fontsize=8)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "02_cross_farm_horizons.png"), dpi=150, bbox_inches="tight")
    plt.close()

# ============================================================================
# MAIN
# ============================================================================

def main():
    start = datetime.now()

    print("=" * 80)
    print("PRE-COMPUTING NORMAL BASELINES")
    print("=" * 80)
    baselines = {}
    for farm in ["A", "B", "C"]:
        mu, sd = compute_farm_baseline(farm, dataset)
        baselines[farm] = (mu, sd)
    print()

    directions = [
        ("A", "C"),
        ("C", "A"),
        ("C", "B"),
        ("B", "A"),
        ("B", "C"),
    ]

    all_results = []
    all_metrics = {}
    summary_rows = []

    for source_farm, target_farm in directions:
        print("=" * 80)
        print(f"TRANSFER: {source_farm} -> {target_farm}")
        print("=" * 80)

        X_train, y_train, source_label_map, source_reverse_map = build_source_training_matrix(source_farm)
        if X_train is None:
            print("  No source matrix built. Skipping.")
            continue

        source_mu, source_sd = baselines[source_farm]
        X_train = zscore_with_baseline(X_train, source_mu, source_sd)

        clf, remap, reverse_remap = train_cross_farm_model(X_train, y_train)
        del X_train, y_train
        gc.collect()

        target_mu, target_sd = baselines[target_farm]
        results = evaluate_transfer(
            source_farm=source_farm,
            target_farm=target_farm,
            clf=clf,
            remap=remap,
            reverse_remap=reverse_remap,
            source_label_map=source_label_map,
            source_reverse_map=source_reverse_map,
            target_mu=target_mu,
            target_sd=target_sd,
        )

        metrics = compute_transfer_metrics(results, source_farm, target_farm)
        key = f"{source_farm}_to_{target_farm}"
        all_metrics[key] = metrics
        all_results.extend(results)

        comp = metrics.get("comparable_faults") or {}
        allf = metrics.get("all_faults") or {}

        summary_rows.append({
            "direction": f"{source_farm}->{target_farm}",
            "source_farm": source_farm,
            "target_farm": target_farm,
            "comparable_detected": comp.get("detected"),
            "comparable_total": comp.get("total"),
            "comparable_detection_rate": comp.get("detection_rate"),
            "comparable_avg_horizon_days": comp.get("avg_horizon_days"),
            "all_detected": allf.get("detected"),
            "all_total": allf.get("total"),
            "all_detection_rate": allf.get("detection_rate"),
            "all_avg_horizon_days": allf.get("avg_horizon_days"),
        })

        print("\nSummary:")
        print(f"  Comparable faults: {(comp.get('detected', 0))}/{(comp.get('total', 0))}  rate={comp.get('detection_rate', 0)}")
        print(f"  All faults       : {(allf.get('detected', 0))}/{(allf.get('total', 0))}  rate={allf.get('detection_rate', 0)}")
        print()

        del clf
        gc.collect()

    print("=" * 80)
    print("SAVING OUTPUTS")
    print("=" * 80)

    results_json = {
        "timestamp": datetime.now().isoformat(),
        "script": "script_3_cross_farm_validation_clean",
        "base_path": BASE_PATH,
        "parameters": {
            "consecutive_window": CONSECUTIVE_WINDOW,
            "confidence_threshold": CONFIDENCE_THRESHOLD,
            "max_class_weight": MAX_CLASS_WEIGHT,
            "cross_farm_groups": CROSS_FARM_GROUPS,
            "roll_windows": ROLL_WINDOWS,
            "normal_status": sorted(list(NORMAL_STATUS)),
            "comparable_fault_types": sorted(list(COMPARABLE_FAULT_TYPES)),
        },
        "metrics": all_metrics,
        "results": all_results,
    }

    json_path = os.path.join(OUTPUT_DIR, "results_cross_farm.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(json_safe(results_json), f, indent=2)

    summary_csv = os.path.join(REPORT_DIR, "cross_farm_summary.csv")
    pd.DataFrame(summary_rows).to_csv(summary_csv, index=False)

    detail_csv = os.path.join(OUTPUT_DIR, "cross_farm_event_results.csv")
    pd.DataFrame(all_results).to_csv(detail_csv, index=False)

    plot_transfer_summary(all_metrics)

    print(f"Saved: {json_path}")
    print(f"Saved: {summary_csv}")
    print(f"Saved: {detail_csv}")
    print(f"Plots: {PLOT_DIR}")

    print("\n" + "=" * 80)
    print("SCRIPT 3 COMPLETE")
    print("=" * 80)
    for row in summary_rows:
        print(
            f"{row['direction']}: "
            f"comparable={row['comparable_detected']}/{row['comparable_total']} "
            f"({row['comparable_detection_rate']}) | "
            f"all={row['all_detected']}/{row['all_total']} "
            f"({row['all_detection_rate']})"
        )
    print(f"\nRuntime: {datetime.now() - start}")

if __name__ == "__main__":
    main()

SCRIPT 3 — CROSS-FARM VALIDATION (CLEAN REBUILD)
Base directory: E:\project\CARE_To_Compare

Farm A registry:
  Column mappings: 81
  electrical             sensors= 14  cols= 14  usable= 14
  temperature            sensors= 11  cols= 11  usable= 11
  generator_other        sensors=  7  cols=  7  usable=  7
  angle                  sensors=  4  cols=  4  usable=  0
  generator_stator       sensors=  3  cols=  3  usable=  3
  transformer            sensors=  3  cols=  3  usable=  3
  environment            sensors=  2  cols=  2  usable=  2
  generator_bearing      sensors=  2  cols=  2  usable=  2
  other                  sensors=  2  cols=  2  usable=  2
  control                sensors=  1  cols=  1  usable=  1
  cooling_water          sensors=  1  cols=  1  usable=  1
  gearbox_bearing        sensors=  1  cols=  1  usable=  1
  gearbox_other          sensors=  1  cols=  1  usable=  1
  hydraulic              sensors=  1  cols=  1  usable=  1
  wind                   sensors=  1  cols

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# SCRIPT 4 — OPERATIONAL ANALYSIS (FINAL CORRECTED VERSION)
# Matches your actual Stage 1 and Stage 2 output column names
# Base path: E:\project\CARE_To_Compare
# ============================================================================

from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================================
# CONFIG
# ============================================================================

BASE_DIR = Path(r"E:\project\CARE_To_Compare")

# Keep the old output structure, just update the root path as requested
STAGE1_EVENT_CSV = BASE_DIR / "pipeline_final_v5" / "event_results_final.csv"
STAGE1_RESULTS_JSON = BASE_DIR / "pipeline_final_v5" / "results_final.json"

STAGE2_RESULTS_CSV = BASE_DIR / "fault_classification_outputs" / "fault_classification_results.csv"
STAGE2_RESULTS_JSON = BASE_DIR / "fault_classification_outputs" / "fault_classification_results.json"

STAGE3_RESULTS_JSON = BASE_DIR / "cross_farm_outputs" / "cross_farm_results.json"

OUTPUT_DIR = BASE_DIR / "operational_analysis_outputs"
REPORTS_DIR = OUTPUT_DIR / "reports"
PLOTS_DIR = OUTPUT_DIR / "plots"

for d in [OUTPUT_DIR, REPORTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# ============================================================================
# HELPERS
# ============================================================================

def safe_read_csv(path: Path) -> pd.DataFrame | None:
    if path.exists():
        return pd.read_csv(path)
    return None


def safe_read_json(path: Path):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None


def json_safe(obj):
    if isinstance(obj, dict):
        return {k: json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_safe(v) for v in obj]
    if isinstance(obj, set):
        return sorted(list(obj))
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return str(obj)
    if pd.isna(obj) if not isinstance(obj, (list, dict, tuple, set, np.ndarray)) else False:
        return None
    return obj


def label_is_fault(v) -> bool:
    s = str(v).strip().lower()
    return s in {"anomaly", "fault", "1", "true", "yes"}


def ensure_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


# ============================================================================
# LOADERS
# ============================================================================

def load_stage1():
    df = safe_read_csv(STAGE1_EVENT_CSV)
    js = safe_read_json(STAGE1_RESULTS_JSON)
    return df, js


def load_stage2():
    df = safe_read_csv(STAGE2_RESULTS_CSV)
    js = safe_read_json(STAGE2_RESULTS_JSON)
    return df, js


def load_stage3():
    js = safe_read_json(STAGE3_RESULTS_JSON)
    return js


# ============================================================================
# STAGE 1 SUMMARY
# Actual columns:
# ['event_id', 'farm', 'label', 'description', 'alarm_raised', 'alarm_source',
#  'max_criticality', 'crit_global', 'crit_subsystem', 'anomaly_rate',
#  'earliness', 'top_subsystem', 'n_pred_rows']
# ============================================================================

def build_stage1_summary(stage1_df: pd.DataFrame, stage1_json=None) -> pd.DataFrame:
    if stage1_df is None or stage1_df.empty:
        return pd.DataFrame()

    df = stage1_df.copy()
    df = ensure_numeric(df, ["alarm_raised", "anomaly_rate", "earliness"])

    df["is_fault_event"] = df["label"].apply(label_is_fault)
    df["is_normal_event"] = ~df["is_fault_event"]
    df["alarm_raised"] = df["alarm_raised"].fillna(0).astype(int)

    rows = []
    for farm, g in df.groupby("farm"):
        anomaly = g[g["is_fault_event"]]
        normal = g[g["is_normal_event"]]

        anomaly_events = len(anomaly)
        normal_events = len(normal)

        detected_events = int(anomaly["alarm_raised"].sum()) if anomaly_events else 0
        false_alarm_events = int(normal["alarm_raised"].sum()) if normal_events else 0

        stage1_detection_rate = detected_events / anomaly_events if anomaly_events else np.nan
        stage1_false_alarm_rate = false_alarm_events / normal_events if normal_events else np.nan

        mean_anomaly_rate_fault = anomaly["anomaly_rate"].mean() if "anomaly_rate" in anomaly.columns else np.nan
        mean_anomaly_rate_normal = normal["anomaly_rate"].mean() if "anomaly_rate" in normal.columns else np.nan
        care_earliness = anomaly.loc[anomaly["alarm_raised"] == 1, "earliness"].mean() if anomaly_events else np.nan

        care_score = np.nan
        care_cov = np.nan
        care_acc = np.nan
        care_rel = np.nan
        care_ear = np.nan

        if isinstance(stage1_json, dict):
            care_block = None

            if "care_scores" in stage1_json and isinstance(stage1_json["care_scores"], dict):
                care_block = stage1_json["care_scores"].get(str(farm))
            elif farm in stage1_json and isinstance(stage1_json[farm], dict):
                care_block = stage1_json[farm]

            if isinstance(care_block, dict):
                care_score = care_block.get("CARE", care_block.get("care_score", np.nan))
                care_cov = care_block.get("coverage", care_block.get("cov", np.nan))
                care_acc = care_block.get("accuracy", care_block.get("acc", np.nan))
                care_rel = care_block.get("reliability", care_block.get("rel", np.nan))
                care_ear = care_block.get("earliness", care_block.get("ear", np.nan))

        rows.append({
            "farm": farm,
            "anomaly_events": anomaly_events,
            "normal_events": normal_events,
            "detected_events": detected_events,
            "false_alarm_events": false_alarm_events,
            "stage1_detection_rate": stage1_detection_rate,
            "stage1_false_alarm_rate": stage1_false_alarm_rate,
            "mean_anomaly_rate_fault": mean_anomaly_rate_fault,
            "mean_anomaly_rate_normal": mean_anomaly_rate_normal,
            "care_earliness_detected_only": care_earliness,
            "care_score": care_score,
            "care_coverage": care_cov,
            "care_accuracy": care_acc,
            "care_reliability": care_rel,
            "care_earliness": care_ear,
        })

    return pd.DataFrame(rows).sort_values("farm").reset_index(drop=True)


# ============================================================================
# STAGE 2 SUMMARY
# Actual columns:
# ['event_id', 'farm', 'fault_type', 'pred_type', 'label_int', 'is_fault',
#  'detected', 'detected_window', 'horizon_days', 'mean_confidence',
#  'uncertain_frac', 'n_train_events', 'n_classes_train',
#  'n_selected_cols', 'sensor_groups']
# ============================================================================

def build_stage2_summary(stage2_df: pd.DataFrame):
    if stage2_df is None or stage2_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    df = stage2_df.copy()
    df = ensure_numeric(
        df,
        [
            "is_fault", "detected", "detected_window", "horizon_days",
            "mean_confidence", "uncertain_frac", "n_train_events",
            "n_classes_train", "n_selected_cols"
        ]
    )

    df["is_fault"] = df["is_fault"].fillna(0).astype(int)
    df["detected"] = df["detected"].fillna(0).astype(int)
    df["detected_window"] = df["detected_window"].fillna(0).astype(int)

    # only anomaly events count for classification performance
    fault_df = df[df["is_fault"] == 1].copy()

    if fault_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    fault_df["correct_class"] = (fault_df["fault_type"].astype(str) == fault_df["pred_type"].astype(str)).astype(int)

    farm_rows = []
    for farm, g in fault_df.groupby("farm"):
        total_faults = len(g)
        detected_5 = int(g["detected"].sum())
        detected_window = int(g["detected_window"].sum())
        correctly_classified = int(g["correct_class"].sum())

        farm_rows.append({
            "farm": farm,
            "stage2_fault_events": total_faults,
            "stage2_detected_5consec": detected_5,
            "stage2_detected_window": detected_window,
            "stage2_detection_rate_5consec": detected_5 / total_faults if total_faults else np.nan,
            "stage2_detection_rate_window": detected_window / total_faults if total_faults else np.nan,
            "stage2_classification_accuracy": correctly_classified / total_faults if total_faults else np.nan,
            "stage2_mean_horizon_days": g.loc[g["detected"] == 1, "horizon_days"].mean(),
            "stage2_mean_confidence": g["mean_confidence"].mean(),
            "stage2_mean_uncertain_frac": g["uncertain_frac"].mean(),
            "stage2_mean_selected_cols": g["n_selected_cols"].mean(),
            "stage2_mean_train_events": g["n_train_events"].mean(),
            "stage2_mean_train_classes": g["n_classes_train"].mean(),
        })

    fault_rows = []
    for (farm, fault_type), g in fault_df.groupby(["farm", "fault_type"]):
        total = len(g)
        det = int(g["detected"].sum())
        det_win = int(g["detected_window"].sum())
        cls = int(g["correct_class"].sum())

        fault_rows.append({
            "farm": farm,
            "fault_type": fault_type,
            "total_events": total,
            "detected_5consec": det,
            "detected_window": det_win,
            "correct_classifications": cls,
            "detection_rate_5consec": det / total if total else np.nan,
            "detection_rate_window": det_win / total if total else np.nan,
            "classification_accuracy": cls / total if total else np.nan,
            "avg_horizon_days": g.loc[g["detected"] == 1, "horizon_days"].mean(),
            "avg_confidence": g["mean_confidence"].mean(),
        })

    farm_summary = pd.DataFrame(farm_rows).sort_values("farm").reset_index(drop=True)
    fault_summary = pd.DataFrame(fault_rows).sort_values(["farm", "fault_type"]).reset_index(drop=True)
    return farm_summary, fault_summary


# ============================================================================
# STAGE 3 SUMMARY
# ============================================================================

def build_stage3_summary(stage3_json) -> pd.DataFrame:
    if stage3_json is None:
        return pd.DataFrame()

    rows = []

    # flexible parsing for different JSON layouts
    if isinstance(stage3_json, dict):
        # expected keys may be "results", "directions", or direct direction keys
        candidate = None

        for k in ["results", "directions", "transfers"]:
            if k in stage3_json and isinstance(stage3_json[k], dict):
                candidate = stage3_json[k]
                break

        if candidate is None:
            candidate = stage3_json

        for direction, block in candidate.items():
            if not isinstance(block, dict):
                continue

            comparable_detected = (
                block.get("comparable_detected")
                or block.get("detected_comparable")
                or block.get("comparable_hits")
                or 0
            )
            comparable_total = (
                block.get("comparable_total")
                or block.get("n_comparable")
                or block.get("comparable_events")
                or 0
            )
            all_detected = (
                block.get("all_detected")
                or block.get("detected_all")
                or block.get("all_hits")
                or 0
            )
            all_total = (
                block.get("all_total")
                or block.get("n_all")
                or block.get("all_events")
                or 0
            )

            comparable_rate = block.get("comparable_rate", np.nan)
            all_rate = block.get("all_rate", np.nan)
            avg_horizon = block.get("avg_horizon_days", block.get("avg_horizon", np.nan))

            if pd.isna(comparable_rate) and comparable_total:
                comparable_rate = comparable_detected / comparable_total
            if pd.isna(all_rate) and all_total:
                all_rate = all_detected / all_total

            rows.append({
                "direction": direction,
                "comparable_detected": comparable_detected,
                "comparable_total": comparable_total,
                "comparable_rate": comparable_rate,
                "all_detected": all_detected,
                "all_total": all_total,
                "all_rate": all_rate,
                "avg_horizon_days": avg_horizon,
            })

    return pd.DataFrame(rows)


# ============================================================================
# FARM-LEVEL MASTER SUMMARY
# ============================================================================

def build_farm_level_summary(stage1_df: pd.DataFrame | None, stage2_df: pd.DataFrame | None) -> pd.DataFrame:
    farms = set()

    if stage1_df is not None and not stage1_df.empty and "farm" in stage1_df.columns:
        farms.update(stage1_df["farm"].dropna().astype(str).unique())

    if stage2_df is not None and not stage2_df.empty and "farm" in stage2_df.columns:
        farms.update(stage2_df["farm"].dropna().astype(str).unique())

    return pd.DataFrame({"farm": sorted(farms)})


def merge_master_summary(farm_ops, stage1_summary, stage2_summary):
    out = farm_ops.copy()

    if not stage1_summary.empty and "farm" in stage1_summary.columns:
        out = out.merge(stage1_summary, on="farm", how="left")

    if not stage2_summary.empty and "farm" in stage2_summary.columns:
        out = out.merge(stage2_summary, on="farm", how="left")

    return out.sort_values("farm").reset_index(drop=True)


# ============================================================================
# PLOTS
# ============================================================================

def plot_stage1_care(stage1_summary: pd.DataFrame):
    if stage1_summary.empty:
        return

    use = stage1_summary.dropna(subset=["care_score"]).copy()
    if use.empty:
        return

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(use["farm"], use["care_score"])
    ax.set_title("Stage 1 CARE Score by Farm")
    ax.set_xlabel("Farm")
    ax.set_ylabel("CARE score")
    ax.set_ylim(0, max(1.0, use["care_score"].max() * 1.15))
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "stage1_care_score_by_farm.png", dpi=300)
    plt.close(fig)


def plot_stage1_detection_false_alarm(stage1_summary: pd.DataFrame):
    if stage1_summary.empty:
        return

    use = stage1_summary.copy()

    x = np.arange(len(use))
    w = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - w/2, use["stage1_detection_rate"].fillna(0), width=w, label="Detection rate")
    ax.bar(x + w/2, use["stage1_false_alarm_rate"].fillna(0), width=w, label="False alarm rate")
    ax.set_xticks(x)
    ax.set_xticklabels(use["farm"])
    ax.set_ylim(0, 1)
    ax.set_title("Stage 1 Detection and False Alarm Rates")
    ax.set_xlabel("Farm")
    ax.set_ylabel("Rate")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "stage1_detection_false_alarm_rates.png", dpi=300)
    plt.close(fig)


def plot_stage2_detection(stage2_farm_summary: pd.DataFrame):
    if stage2_farm_summary.empty:
        return

    use = stage2_farm_summary.copy()

    x = np.arange(len(use))
    w = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - w/2, use["stage2_detection_rate_5consec"].fillna(0), width=w, label="5-consecutive")
    ax.bar(x + w/2, use["stage2_classification_accuracy"].fillna(0), width=w, label="Classification accuracy")
    ax.set_xticks(x)
    ax.set_xticklabels(use["farm"])
    ax.set_ylim(0, 1)
    ax.set_title("Stage 2 Detection and Classification Accuracy")
    ax.set_xlabel("Farm")
    ax.set_ylabel("Rate")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "stage2_detection_classification_by_farm.png", dpi=300)
    plt.close(fig)


def plot_stage2_horizon(stage2_fault_summary: pd.DataFrame):
    if stage2_fault_summary.empty:
        return

    use = stage2_fault_summary.dropna(subset=["avg_horizon_days"]).copy()
    if use.empty:
        return

    use["farm_fault"] = use["farm"].astype(str) + " — " + use["fault_type"].astype(str)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(use["farm_fault"], use["avg_horizon_days"])
    ax.set_title("Average Warning Horizon by Farm and Fault Type")
    ax.set_xlabel("Farm / Fault type")
    ax.set_ylabel("Average horizon (days)")
    ax.tick_params(axis="x", rotation=75)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "stage2_horizon_by_farm_fault.png", dpi=300)
    plt.close(fig)


def plot_cross_farm(stage3_summary: pd.DataFrame):
    if stage3_summary.empty:
        return

    use = stage3_summary.copy()

    x = np.arange(len(use))
    w = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - w/2, use["comparable_rate"].fillna(0), width=w, label="Comparable faults")
    ax.bar(x + w/2, use["all_rate"].fillna(0), width=w, label="All faults")
    ax.set_xticks(x)
    ax.set_xticklabels(use["direction"], rotation=30, ha="right")
    ax.set_ylim(0, 1)
    ax.set_title("Cross-Farm Transfer Detection Rates")
    ax.set_xlabel("Transfer direction")
    ax.set_ylabel("Rate")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "cross_farm_detection_rates.png", dpi=300)
    plt.close(fig)


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("=" * 80)
    print("SCRIPT 4 — OPERATIONAL ANALYSIS (FINAL CORRECTED VERSION)")
    print("=" * 80)
    print(f"Base directory: {BASE_DIR}")
    print()

    print("Loading Stage 1 outputs...")
    stage1_df, stage1_json = load_stage1()
    if stage1_df is None:
        print("  Stage 1 CSV not found.")
    else:
        print(f"  Stage 1 rows: {len(stage1_df)}")
        print(f"  Stage 1 columns: {list(stage1_df.columns)}")

    print()
    print("Loading Stage 2 outputs...")
    stage2_df, stage2_json = load_stage2()
    if stage2_df is None:
        print("  Stage 2 CSV not found.")
    else:
        print(f"  Stage 2 rows: {len(stage2_df)}")
        print(f"  Stage 2 columns: {list(stage2_df.columns)}")

    print()
    print("Loading Stage 3 outputs...")
    stage3_json = load_stage3()
    if stage3_json is None:
        print("  Stage 3 JSON not found.")
    else:
        print("  Stage 3 JSON loaded.")

    print()
    print("Building summaries...")
    farm_ops_summary = build_farm_level_summary(stage1_df, stage2_df)
    stage1_summary = build_stage1_summary(stage1_df, stage1_json) if stage1_df is not None else pd.DataFrame()
    stage2_farm_summary, stage2_fault_summary = build_stage2_summary(stage2_df) if stage2_df is not None else (pd.DataFrame(), pd.DataFrame())
    stage3_summary = build_stage3_summary(stage3_json)

    master_summary = merge_master_summary(farm_ops_summary, stage1_summary, stage2_farm_summary)

    print(f"  farm_ops_summary columns: {list(farm_ops_summary.columns)}")
    print(f"  stage1_summary columns: {list(stage1_summary.columns)}")
    print(f"  stage2_farm_summary columns: {list(stage2_farm_summary.columns)}")
    print(f"  stage3_summary columns: {list(stage3_summary.columns)}")
    print()

    print("Saving reports...")
    if not master_summary.empty:
        master_summary.to_csv(REPORTS_DIR / "master_farm_summary.csv", index=False)
        print(f"  Saved: {REPORTS_DIR / 'master_farm_summary.csv'}")

    if not stage1_summary.empty:
        stage1_summary.to_csv(REPORTS_DIR / "stage1_summary.csv", index=False)
        print(f"  Saved: {REPORTS_DIR / 'stage1_summary.csv'}")

    if not stage2_farm_summary.empty:
        stage2_farm_summary.to_csv(REPORTS_DIR / "stage2_farm_summary.csv", index=False)
        print(f"  Saved: {REPORTS_DIR / 'stage2_farm_summary.csv'}")

    if not stage2_fault_summary.empty:
        stage2_fault_summary.to_csv(REPORTS_DIR / "stage2_fault_summary.csv", index=False)
        print(f"  Saved: {REPORTS_DIR / 'stage2_fault_summary.csv'}")

    if not stage3_summary.empty:
        stage3_summary.to_csv(REPORTS_DIR / "cross_farm_summary.csv", index=False)
        print(f"  Saved: {REPORTS_DIR / 'cross_farm_summary.csv'}")

    manifest = {
        "timestamp": datetime.now().isoformat(),
        "script": "script_4_operational_analysis_final_corrected",
        "base_dir": str(BASE_DIR),
        "inputs": {
            "stage1_event_csv": str(STAGE1_EVENT_CSV),
            "stage1_results_json": str(STAGE1_RESULTS_JSON),
            "stage2_results_csv": str(STAGE2_RESULTS_CSV),
            "stage2_results_json": str(STAGE2_RESULTS_JSON),
            "stage3_results_json": str(STAGE3_RESULTS_JSON),
        },
        "outputs": {
            "output_dir": str(OUTPUT_DIR),
            "reports_dir": str(REPORTS_DIR),
            "plots_dir": str(PLOTS_DIR),
        },
        "row_counts": {
            "stage1_rows": 0 if stage1_df is None else int(len(stage1_df)),
            "stage2_rows": 0 if stage2_df is None else int(len(stage2_df)),
            "stage1_summary_rows": int(len(stage1_summary)),
            "stage2_farm_summary_rows": int(len(stage2_farm_summary)),
            "stage2_fault_summary_rows": int(len(stage2_fault_summary)),
            "stage3_summary_rows": int(len(stage3_summary)),
        },
    }

    with open(OUTPUT_DIR / "operational_analysis_manifest.json", "w", encoding="utf-8") as f:
        json.dump(json_safe(manifest), f, indent=2)
    print(f"  Saved: {OUTPUT_DIR / 'operational_analysis_manifest.json'}")

    print()
    print("Generating plots...")
    plot_stage1_care(stage1_summary)
    plot_stage1_detection_false_alarm(stage1_summary)
    plot_stage2_detection(stage2_farm_summary)
    plot_stage2_horizon(stage2_fault_summary)
    plot_cross_farm(stage3_summary)
    print(f"  Plots saved to: {PLOTS_DIR}")

    print()
    print("=" * 80)
    print("SCRIPT 4 COMPLETE")
    print("=" * 80)
    print(f"Outputs -> {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

SCRIPT 4 — OPERATIONAL ANALYSIS (FINAL CORRECTED VERSION)
Base directory: E:\project\CARE_To_Compare

Loading Stage 1 outputs...
  Stage 1 rows: 95
  Stage 1 columns: ['event_id', 'farm', 'label', 'description', 'alarm_raised', 'alarm_source', 'max_criticality', 'crit_global', 'crit_subsystem', 'anomaly_rate', 'earliness', 'top_subsystem', 'n_pred_rows']

Loading Stage 2 outputs...
  Stage 2 rows: 95
  Stage 2 columns: ['event_id', 'farm', 'fault_type', 'pred_type', 'label_int', 'is_fault', 'detected', 'detected_window', 'horizon_days', 'mean_confidence', 'uncertain_frac', 'n_train_events', 'n_classes_train', 'n_selected_cols', 'sensor_groups']

Loading Stage 3 outputs...
  Stage 3 JSON loaded.

Building summaries...
  farm_ops_summary columns: ['farm']
  stage1_summary columns: ['farm', 'anomaly_events', 'normal_events', 'detected_events', 'false_alarm_events', 'stage1_detection_rate', 'stage1_false_alarm_rate', 'mean_anomaly_rate_fault', 'mean_anomaly_rate_normal', 'care_earliness_de

SCRIPT 5 — DISSERTATION FIGURES (FAULT-TYPE-AWARE, FULL VERSION)
Base directory: E:\project\CARE_To_Compare

Loading outputs...
  Stage 1 CSV : E:\project\CARE_To_Compare\pipeline_final_v5\event_results_final.csv
  Stage 2 CSV : E:\project\CARE_To_Compare\fault_classification_outputs\fault_classification_results.csv
  Stage 1 JSON: E:\project\CARE_To_Compare\pipeline_final_v5\results_final.json
  Stage 3 JSON: E:\project\CARE_To_Compare\cross_farm_outputs\cross_farm_results.json
  Stage 1 events : 95
  Stage 2 events : 95
  Stage 3 summary: 5

Generating performance figures...
  Saved: E:\project\CARE_To_Compare\dissertation_figures_outputs\plots\01_care_score_by_farm.png
  Saved: E:\project\CARE_To_Compare\dissertation_figures_outputs\plots\02_care_component_breakdown.png
  Saved: E:\project\CARE_To_Compare\dissertation_figures_outputs\plots\03_stage2_performance_by_farm.png
  Saved: E:\project\CARE_To_Compare\dissertation_figures_outputs\plots\05_stage2_selected_columns_by_farm.png
 

In [3]:
# script_5_dissertation_figures.py
# ============================================================================
# SCRIPT 5 — DISSERTATION FIGURES (FAULT-TYPE-AWARE, SHARED-SENSOR VERSION)
# ============================================================================
# Uses:
#   - Stage 1 from pipeline_final_v5
#   - Stage 2 from fault_classification_outputs
#   - Stage 3 from cross_farm_outputs
#   - Raw CARE event CSVs from Wind Farm A/B/C datasets folders
#   - feature_description.csv for human-readable sensor names
#
# Key correction:
#   For each farm, the FAULT event determines the sensor list.
#   The HEALTHY comparison event reuses the EXACT SAME sensor list.
#   So healthy vs fault is a true state comparison, not different variables.
# ============================================================================

from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================================
# CONFIG
# ============================================================================

BASE_DIR = Path(r"E:\project\CARE_To_Compare")

STAGE1_EVENT_CSV = BASE_DIR / "pipeline_final_v5" / "event_results_final.csv"
STAGE1_JSON = BASE_DIR / "pipeline_final_v5" / "results_final.json"
STAGE1_SUMMARY_CSV = BASE_DIR / "pipeline_final_v5" / "reports" / "care_score_summary.csv"

STAGE2_CSV = BASE_DIR / "fault_classification_outputs" / "fault_classification_results.csv"

STAGE3_JSON = BASE_DIR / "cross_farm_outputs" / "cross_farm_results.json"
STAGE3_ALT_JSON = BASE_DIR / "cross_farm_outputs" / "results_cross_farm.json"
STAGE3_SUMMARY_CSV = BASE_DIR / "cross_farm_outputs" / "reports" / "cross_farm_summary.csv"

FARM_METADATA_PATHS = {
    "A": BASE_DIR / "Wind Farm A" / "feature_description.csv",
    "B": BASE_DIR / "Wind Farm B" / "feature_description.csv",
    "C": BASE_DIR / "Wind Farm C" / "feature_description.csv",
}

OUTPUT_DIR = BASE_DIR / "dissertation_figures_outputs"
PLOTS_DIR = OUTPUT_DIR / "plots"
REPORTS_DIR = OUTPUT_DIR / "reports"

RAW_SEARCH_ROOTS = [
    BASE_DIR,
    BASE_DIR / "Wind Farm A",
    BASE_DIR / "Wind Farm B",
    BASE_DIR / "Wind Farm C",
]

FARMS = ["A", "B", "C"]

# Prefer strong, dissertation-friendly examples
PREFERRED_EXAMPLES = {
    "A": {
        "fault": ["E72", "E51", "E45", "E84", "E68", "E40"],
        "healthy": ["E69", "E25", "E13", "E24"],
    },
    "B": {
        "fault": ["E34", "E7", "E19", "E27", "E53", "E77"],
        "healthy": ["E52", "E83", "E21", "E23"],
    },
    "C": {
        "fault": ["E11", "E16", "E55", "E66", "E70", "E81", "E47"],
        "healthy": ["E80", "E8", "E85", "E6"],
    },
}

# Fault-type-aware keyword guidance
FAULT_GROUP_RULES = {
    "gearbox": ["gearbox", "oil", "bearing", "planetary", "lubric", "main shaft"],
    "gearbox_oil": ["gearbox", "oil", "lubric", "pressure", "temperature"],
    "generator_bearing": ["generator", "bearing", "drive-end", "non-drive", "nde", "de"],
    "generator_stator": ["stator", "generator", "winding", "phase", "temperature"],
    "transformer": ["transformer", "winding", "insulation", "phase", "electrical", "voltage", "current"],
    "hydraulic": ["hydraulic", "pressure", "pump", "oil", "brake", "azimuth"],
    "rotor_bearing": ["rotor bearing", "main bearing", "bearing", "shaft"],
    "pitch_electrical": ["pitch", "battery", "capacitor", "dc-link", "motor", "voltage", "electrical"],
    "pitch_mechanical": ["pitch", "bearing", "gearbox", "mechanical", "angle", "position"],
    "converter": ["converter", "igbt", "dc-link", "torque", "current", "voltage"],
    "communication": ["communication", "beckhoff", "network", "controller", "fieldbus", "ethercat"],
    "cooling_water": ["cooling", "water", "stator cooler", "flow meter", "pump"],
    "yaw": ["yaw", "drive", "motor", "angle", "position"],
    "electrical_internal": ["electrical", "voltage", "current", "phase", "internal"],
    "rotor_brake": ["brake", "rotor", "hydraulic", "pressure", "temperature"],
    "normal": [],
}

# Generic channels to down-rank when better fault-specific signals exist
GENERIC_POWER_WORDS = [
    "total active power",
    "total reactive power",
    "active power",
    "reactive power",
    "power ",
    "energy ",
    "kwh",
    "kvarh",
]


# ============================================================================
# IO HELPERS
# ============================================================================

def ensure_dirs():
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)


def safe_read_csv(path: Path, **kwargs) -> Optional[pd.DataFrame]:
    attempts = [
        {"sep": ";", "encoding": "utf-8"},
        {"sep": ";", "encoding": "latin1"},
        {"sep": ",", "encoding": "utf-8"},
        {"sep": ",", "encoding": "latin1"},
    ]

    best_df = None
    best_score = -1

    for attempt in attempts:
        try:
            params = dict(kwargs)
            params.update(attempt)
            df = pd.read_csv(path, **params)
            df.columns = [str(c).strip() for c in df.columns]
            score = df.shape[1]
            if score > best_score:
                best_df = df
                best_score = score
        except Exception:
            continue

    return best_df


def load_json(path: Path) -> Optional[dict]:
    if not path.exists():
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def save_fig(fig, filename: str):
    out = PLOTS_DIR / filename
    fig.tight_layout()
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {out}")


def json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_safe(v) for v in obj]
    if isinstance(obj, set):
        return sorted(list(obj))
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return str(obj)
    return obj


# ============================================================================
# TEXT / NORMALISERS
# ============================================================================

def norm_event_id(x) -> str:
    s = str(x).strip().upper()
    m = re.search(r"(\d+)", s)
    return f"E{int(m.group(1))}" if m else s


def norm_farm(x) -> str:
    s = str(x).strip().upper()
    m = re.search(r"([ABC])", s)
    return m.group(1) if m else s


def clean_text(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def clean_unit_text(unit: str) -> str:
    if unit is None or (isinstance(unit, float) and pd.isna(unit)):
        return ""

    u = str(unit).strip()

    replacements = {
        "�C": "°C",
        "Â°C": "°C",
        "degC": "°C",
        "DegC": "°C",
        "Celsius": "°C",
        "�": "",
        "m2/s2": "m²/s²",
        "m3": "m³",
        "l/h": "l/h",
    }

    for bad, good in replacements.items():
        u = u.replace(bad, good)

    return u


# ============================================================================
# METADATA / SENSOR NAMING
# ============================================================================

def load_feature_description(farm: str) -> pd.DataFrame:
    path = FARM_METADATA_PATHS.get(farm)
    if path is None or not path.exists():
        return pd.DataFrame()

    df = safe_read_csv(path)
    if df is None or df.empty:
        return pd.DataFrame()

    df.columns = [str(c).strip() for c in df.columns]
    return df


def build_sensor_metadata_map(farm: str) -> Dict[str, dict]:
    """
    base sensor name -> metadata dict
    example:
        sensor_57 -> {
            "label": "Main bearing temperature [°C]",
            "description": "...",
            "unit": "°C",
            "is_angle": False,
            "is_counter": False
        }
    """
    df = load_feature_description(farm)
    if df.empty:
        return {}

    lower_cols = {str(c).lower().strip(): c for c in df.columns}

    sensor_col = lower_cols.get("sensor_name")
    desc_col = lower_cols.get("description")
    unit_col = lower_cols.get("unit")
    angle_col = lower_cols.get("is_angle")
    counter_col = lower_cols.get("is_counter")

    if sensor_col is None:
        return {}

    out = {}

    for _, row in df.iterrows():
        raw_sensor = clean_text(row.get(sensor_col, ""))
        if not raw_sensor:
            continue

        desc = clean_text(row.get(desc_col, "")) if desc_col else ""
        unit = clean_unit_text(row.get(unit_col, "")) if unit_col else ""

        label = desc if desc else raw_sensor
        if unit:
            label = f"{label} [{unit}]"

        is_angle = bool(row.get(angle_col, False)) if angle_col else False
        is_counter = bool(row.get(counter_col, False)) if counter_col else False

        out[raw_sensor] = {
            "label": label,
            "description": desc,
            "unit": unit,
            "is_angle": is_angle,
            "is_counter": is_counter,
        }

    return out


def split_sensor_stat(col_name: str) -> Tuple[str, Optional[str]]:
    s = str(col_name).strip()
    m = re.match(r"^(.*)_(avg|max|min|std)$", s, flags=re.IGNORECASE)
    if m:
        return m.group(1), m.group(2).lower()
    return s, None


def pretty_stat_suffix(stat: Optional[str]) -> str:
    if stat is None:
        return ""
    mapping = {
        "avg": "mean",
        "max": "max",
        "min": "min",
        "std": "std",
    }
    return mapping.get(stat, stat)


def pretty_sensor_name(sensor_name: str, farm: str, meta_map: Dict[str, dict]) -> str:
    base, stat = split_sensor_stat(sensor_name)
    md = meta_map.get(base, {})
    base_label = md.get("label", base.replace("_", " "))
    stat_label = pretty_stat_suffix(stat)
    if stat_label:
        return f"{base_label} ({stat_label})"
    return base_label


# ============================================================================
# LOAD STAGE OUTPUTS
# ============================================================================

def load_stage1_event_df() -> pd.DataFrame:
    if STAGE1_EVENT_CSV.exists():
        print(f"  Stage 1 CSV : {STAGE1_EVENT_CSV}")
        return pd.read_csv(STAGE1_EVENT_CSV)
    print("  Stage 1 CSV not found.")
    return pd.DataFrame()


def normalise_stage1_event_df(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]

    if "event_id" in out.columns:
        out["event_id"] = out["event_id"].map(norm_event_id)
    if "farm" in out.columns:
        out["farm"] = out["farm"].map(norm_farm)

    if "label" in out.columns:
        out["label"] = out["label"].astype(str).str.lower()
        out["is_fault"] = (out["label"] != "normal").astype(int)
    else:
        out["is_fault"] = 0

    if "alarm_raised" in out.columns:
        out["alarm_raised"] = pd.to_numeric(out["alarm_raised"], errors="coerce").fillna(0).astype(int)

    for c in ["anomaly_rate", "earliness", "max_criticality", "crit_global", "crit_subsystem", "n_pred_rows"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    return out


def load_stage1_summary() -> pd.DataFrame:
    if STAGE1_SUMMARY_CSV.exists():
        try:
            df = pd.read_csv(STAGE1_SUMMARY_CSV)
            df.columns = [str(c).strip() for c in df.columns]
            if "farm" in df.columns:
                df["farm"] = df["farm"].map(norm_farm)
            return df
        except Exception:
            pass

    j = load_json(STAGE1_JSON)
    if not j:
        return pd.DataFrame()

    rows = []
    if isinstance(j, dict):
        for k, v in j.items():
            if k in FARMS and isinstance(v, dict):
                rows.append({
                    "farm": k,
                    "care_score": v.get("care_score", v.get("care")),
                    "care_coverage": v.get("coverage", v.get("cov")),
                    "care_accuracy": v.get("accuracy", v.get("acc")),
                    "care_reliability": v.get("reliability", v.get("rel")),
                    "care_earliness": v.get("earliness", v.get("ear")),
                })
    return pd.DataFrame(rows)


def load_stage2_df() -> pd.DataFrame:
    if STAGE2_CSV.exists():
        print(f"  Stage 2 CSV : {STAGE2_CSV}")
        return pd.read_csv(STAGE2_CSV)
    print("  Stage 2 CSV not found.")
    return pd.DataFrame()


def normalise_stage2_df(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]

    if "event_id" in out.columns:
        out["event_id"] = out["event_id"].map(norm_event_id)
    if "farm" in out.columns:
        out["farm"] = out["farm"].map(norm_farm)

    for c in [
        "detected", "detected_window", "is_fault", "horizon_days",
        "mean_confidence", "uncertain_frac", "n_selected_cols"
    ]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    if "fault_type" in out.columns:
        out["fault_type"] = out["fault_type"].astype(str).str.strip().str.lower()
    if "pred_type" in out.columns:
        out["pred_type"] = out["pred_type"].astype(str).str.strip().str.lower()

    return out


def load_stage3_summary() -> pd.DataFrame:
    if STAGE3_SUMMARY_CSV.exists():
        try:
            df = pd.read_csv(STAGE3_SUMMARY_CSV)
            df.columns = [str(c).strip() for c in df.columns]
            return df
        except Exception:
            pass

    for p in [STAGE3_JSON, STAGE3_ALT_JSON]:
        j = load_json(p)
        if not j:
            continue

        rows = []
        if isinstance(j, dict):
            if "summary_rows" in j and isinstance(j["summary_rows"], list):
                return pd.DataFrame(j["summary_rows"])

            for k, v in j.items():
                if isinstance(v, dict):
                    rows.append({
                        "direction": v.get("direction", k),
                        "comparable_rate": v.get("comparable_rate"),
                        "all_rate": v.get("all_rate"),
                        "avg_horizon_days": v.get("avg_horizon_days"),
                    })

        if rows:
            return pd.DataFrame(rows)

    return pd.DataFrame()


# ============================================================================
# RAW FILE INDEXING
# ============================================================================

def infer_event_id(path_obj: Path, df: pd.DataFrame) -> Optional[str]:
    txt = str(path_obj).upper()
    m = re.search(r"E\s*0*(\d+)", txt)
    if m:
        return f"E{int(m.group(1))}"

    stem = path_obj.stem.strip().upper()
    if stem.isdigit():
        return f"E{int(stem)}"

    for c in df.columns:
        cl = str(c).lower().strip()
        if cl in {"event_id", "event", "id"}:
            vals = df[c].dropna().astype(str).tolist()
            for v in vals[:20]:
                m2 = re.search(r"(\d+)", v.upper())
                if m2:
                    return f"E{int(m2.group(1))}"
    return None


def infer_farm(path_obj: Path, df: pd.DataFrame) -> Optional[str]:
    txt = str(path_obj).lower()

    if "wind farm a" in txt or "farm a" in txt or "farm_a" in txt:
        return "A"
    if "wind farm b" in txt or "farm b" in txt or "farm_b" in txt:
        return "B"
    if "wind farm c" in txt or "farm c" in txt or "farm_c" in txt:
        return "C"

    ncols = len(df.columns)
    if ncols < 150:
        return "A"
    elif ncols < 500:
        return "B"
    return "C"


def looks_like_event_csv(path: Path) -> bool:
    name = path.name.lower()
    parent_txt = str(path.parent).lower()

    bad_keywords = [
        "event_results", "summary", "registry", "feature_description",
        "fault_classification", "care_score", "manifest", "report",
        "results_final", "cross_farm", "sensor_registry",
    ]
    if any(k in name for k in bad_keywords):
        return False

    if "datasets" in parent_txt and path.stem.isdigit():
        return True

    df = safe_read_csv(path, nrows=20)
    if df is None or df.empty:
        return False

    cols = [str(c).lower().strip() for c in df.columns]
    return "train_test" in cols or len(cols) >= 20


def index_raw_event_files() -> Dict[Tuple[str, str], Path]:
    candidates = []
    for root in RAW_SEARCH_ROOTS:
        if root.exists():
            candidates.extend(root.rglob("*.csv"))

    print(f"  Candidate raw event CSVs found: {len(candidates)}")

    index: Dict[Tuple[str, str], Path] = {}
    for path in candidates:
        if not looks_like_event_csv(path):
            continue

        df = safe_read_csv(path, nrows=20)
        if df is None or df.empty:
            continue

        farm = infer_farm(path, df)
        event_id = infer_event_id(path, df)

        if farm and event_id:
            index[(farm, event_id)] = path

    print(f"  Indexed event files: {len(index)}")
    return index


# ============================================================================
# RAW EVENT PROCESSING
# ============================================================================

def split_train_pred(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if "train_test" not in df.columns:
        return df.copy(), df.copy()

    s = df["train_test"].astype(str).str.lower().str.strip()
    train_df = df[s.eq("train")].copy()
    pred_df = df[s.isin(["pred", "prediction", "predict", "test"])].copy()

    if pred_df.empty:
        pred_df = df[~s.eq("train")].copy()
    if train_df.empty:
        train_df = df.copy()

    return train_df, pred_df


def coerce_numeric_sensor_block(df: pd.DataFrame) -> pd.DataFrame:
    exclude = {
        "train_test", "status_type_id", "status", "event_id", "id",
        "timestamp", "time_stamp", "time", "datetime", "date", "asset_id"
    }

    data = {}
    for c in df.columns:
        cl = str(c).lower().strip()
        if cl in exclude:
            continue

        s = df[c].astype(str).str.strip()
        s = s.str.replace(",", ".", regex=False)
        num = pd.to_numeric(s, errors="coerce")
        valid_ratio = num.notna().mean() if len(num) else 0

        if valid_ratio >= 0.3:
            data[c] = num

    if not data:
        return pd.DataFrame(index=df.index)

    return pd.DataFrame(data, index=df.index)


def normalise_series(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    mu = s.mean()
    sd = s.std()
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mu) / sd


def moving_mean(s: pd.Series, w: int = 6) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    return s.rolling(window=w, min_periods=1).mean()


def score_sensor_shift(train_s: pd.Series, pred_s: pd.Series) -> float:
    train_s = pd.to_numeric(train_s, errors="coerce")
    pred_s = pd.to_numeric(pred_s, errors="coerce")

    train_mu = train_s.mean()
    train_sd = train_s.std()
    pred_mu = pred_s.mean()

    if pd.isna(train_sd) or train_sd == 0:
        return 0.0
    return float(abs((pred_mu - train_mu) / train_sd))


def keyword_score_for_fault(sensor_col: str, fault_type: str, farm: str, meta_map: Dict[str, dict]) -> int:
    base, _ = split_sensor_stat(sensor_col)
    md = meta_map.get(base, {})
    desc = md.get("description", "")
    label = md.get("label", base)
    txt = f"{base} {desc} {label}".lower()

    score = 0
    for kw in FAULT_GROUP_RULES.get(fault_type, []):
        if kw.lower() in txt:
            score += 3

    # down-rank generic power/energy channels
    if any(w in txt for w in GENERIC_POWER_WORDS):
        score -= 2

    # slightly down-rank counters and angle variables
    if md.get("is_counter", False):
        score -= 5
    if md.get("is_angle", False):
        score -= 2

    return score


# ============================================================================
# FAULT-AWARE SELECTION
# ============================================================================

def best_fault_type_for_event(stage2_df: pd.DataFrame, farm: str, event_id: str) -> str:
    if stage2_df.empty or not {"farm", "event_id", "fault_type"}.issubset(stage2_df.columns):
        return "normal"

    m = stage2_df[(stage2_df["farm"] == farm) & (stage2_df["event_id"] == event_id)]
    if m.empty:
        return "normal"

    vals = m["fault_type"].dropna().astype(str).str.lower().tolist()
    if not vals:
        return "normal"

    return vals[0]


def choose_example_event_pairs(stage1_df: pd.DataFrame, stage2_df: pd.DataFrame):
    pairs = {}

    if stage1_df.empty:
        return pairs

    for farm in FARMS:
        sub = stage1_df[stage1_df["farm"] == farm].copy()
        if sub.empty:
            continue

        healthy = sub[sub["is_fault"] == 0].copy()
        faulty = sub[sub["is_fault"] == 1].copy()

        chosen_fault = None
        chosen_fault_type = "normal"
        chosen_healthy = None

        for eid in PREFERRED_EXAMPLES.get(farm, {}).get("fault", []):
            m = faulty[faulty["event_id"] == eid]
            if not m.empty:
                chosen_fault = eid
                chosen_fault_type = best_fault_type_for_event(stage2_df, farm, eid)
                break

        if chosen_fault is None and not faulty.empty:
            sort_cols = [c for c in ["alarm_raised", "anomaly_rate", "max_criticality"] if c in faulty.columns]
            if sort_cols:
                faulty = faulty.sort_values(sort_cols, ascending=[False] * len(sort_cols))
            chosen_fault = faulty.iloc[0]["event_id"]
            chosen_fault_type = best_fault_type_for_event(stage2_df, farm, chosen_fault)

        for eid in PREFERRED_EXAMPLES.get(farm, {}).get("healthy", []):
            m = healthy[healthy["event_id"] == eid]
            if not m.empty:
                chosen_healthy = eid
                break

        if chosen_healthy is None and not healthy.empty:
            sort_cols = [c for c in ["alarm_raised", "anomaly_rate"] if c in healthy.columns]
            if sort_cols:
                healthy = healthy.sort_values(sort_cols, ascending=True)
            chosen_healthy = healthy.iloc[0]["event_id"]

        if chosen_fault is not None and chosen_healthy is not None:
            pairs[farm] = {
                "healthy_event": chosen_healthy,
                "fault_event": chosen_fault,
                "fault_type": chosen_fault_type,
            }

    return pairs


def choose_fault_relevant_columns(
    farm: str,
    fault_type: str,
    train_df: pd.DataFrame,
    pred_df: pd.DataFrame,
    meta_map: Dict[str, dict],
    max_cols: int = 4
) -> List[str]:
    train_num = coerce_numeric_sensor_block(train_df)
    pred_num = coerce_numeric_sensor_block(pred_df)

    shared = [c for c in train_num.columns if c in pred_num.columns]
    if not shared:
        return []

    scored = []
    for c in shared:
        shift = score_sensor_shift(train_num[c], pred_num[c])
        kw_score = keyword_score_for_fault(c, fault_type, farm, meta_map)

        base, _ = split_sensor_stat(c)
        md = meta_map.get(base, {})
        label_txt = md.get("label", base).lower()

        # prefer non-generic channels if fault-specific channels exist
        generic_penalty = 1 if any(w in label_txt for w in GENERIC_POWER_WORDS) else 0

        total_score = (2.5 * shift) + kw_score - generic_penalty

        scored.append({
            "col": c,
            "shift": shift,
            "kw_score": kw_score,
            "total_score": total_score,
            "is_generic": generic_penalty > 0,
        })

    score_df = pd.DataFrame(scored).sort_values(
        ["kw_score", "total_score", "shift"],
        ascending=[False, False, False]
    )

    # Prefer actual fault-relevant channels if any have positive keyword score
    if (score_df["kw_score"] > 0).any():
        score_df = score_df[score_df["kw_score"] > 0].copy()

    # Avoid choosing multiple stats from exactly same base if possible
    selected = []
    used_bases = set()

    for _, row in score_df.iterrows():
        c = row["col"]
        base, _ = split_sensor_stat(c)

        if base not in used_bases:
            selected.append(c)
            used_bases.add(base)

        if len(selected) >= max_cols:
            break

    # fallback if too few
    if len(selected) < max_cols:
        for _, row in score_df.iterrows():
            c = row["col"]
            if c not in selected:
                selected.append(c)
            if len(selected) >= max_cols:
                break

    return selected[:max_cols]


def choose_shared_pair_sensors(
    farm: str,
    fault_event_id: str,
    fault_type: str,
    raw_index: Dict[Tuple[str, str], Path],
    max_cols: int = 4
) -> List[str]:
    raw_path = raw_index.get((farm, fault_event_id))
    if raw_path is None:
        return []

    df = safe_read_csv(raw_path)
    if df is None or df.empty:
        return []

    meta_map = build_sensor_metadata_map(farm)
    train_df, pred_df = split_train_pred(df)

    sensors = choose_fault_relevant_columns(
        farm=farm,
        fault_type=fault_type,
        train_df=train_df if not train_df.empty else df,
        pred_df=pred_df if not pred_df.empty else df,
        meta_map=meta_map,
        max_cols=max_cols
    )
    return sensors


# ============================================================================
# PERFORMANCE PLOTS
# ============================================================================

def plot_care_score_by_farm(summary_df: pd.DataFrame):
    if summary_df.empty:
        return

    farm_col = None
    care_col = None
    for c in summary_df.columns:
        cl = str(c).lower().strip()
        if cl == "farm":
            farm_col = c
        if cl in {"care_score", "care"}:
            care_col = c

    if farm_col is None or care_col is None:
        return

    plot_df = summary_df[[farm_col, care_col]].copy()
    plot_df.columns = ["farm", "care_score"]
    plot_df["farm"] = plot_df["farm"].map(norm_farm)
    plot_df["care_score"] = pd.to_numeric(plot_df["care_score"], errors="coerce")
    plot_df = plot_df.dropna().sort_values("farm")
    if plot_df.empty:
        return

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(plot_df["farm"], plot_df["care_score"])
    ax.set_title("Stage 1 CARE score by farm")
    ax.set_xlabel("Farm")
    ax.set_ylabel("CARE score")
    ax.set_ylim(0, max(0.8, plot_df["care_score"].max() + 0.05))

    for i, v in enumerate(plot_df["care_score"]):
        ax.text(i, v + 0.01, f"{v:.3f}", ha="center", va="bottom")

    save_fig(fig, "01_care_score_by_farm.png")


def plot_care_components(summary_df: pd.DataFrame):
    if summary_df.empty:
        return

    rename = {}
    for c in summary_df.columns:
        cl = str(c).lower().strip()
        if cl == "farm":
            rename[c] = "farm"
        elif cl in {"care_coverage", "coverage", "cov"}:
            rename[c] = "care_coverage"
        elif cl in {"care_accuracy", "accuracy", "acc"}:
            rename[c] = "care_accuracy"
        elif cl in {"care_reliability", "reliability", "rel"}:
            rename[c] = "care_reliability"
        elif cl in {"care_earliness", "earliness", "ear"}:
            rename[c] = "care_earliness"

    plot_df = summary_df.rename(columns=rename).copy()
    cols = ["farm", "care_coverage", "care_accuracy", "care_reliability", "care_earliness"]
    if not all(c in plot_df.columns for c in cols):
        return

    plot_df = plot_df[cols].copy()
    plot_df["farm"] = plot_df["farm"].map(norm_farm)
    for c in cols[1:]:
        plot_df[c] = pd.to_numeric(plot_df[c], errors="coerce")
    plot_df = plot_df.dropna().sort_values("farm")
    if plot_df.empty:
        return

    x = np.arange(len(plot_df))
    width = 0.2
    components = ["care_coverage", "care_accuracy", "care_reliability", "care_earliness"]

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, c in enumerate(components):
        ax.bar(x + (i - 1.5) * width, plot_df[c], width=width, label=c.replace("care_", "").title())

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["farm"])
    ax.set_title("Stage 1 CARE component breakdown")
    ax.set_xlabel("Farm")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.05)
    ax.legend()

    save_fig(fig, "02_care_component_breakdown.png")


def plot_stage2_performance(stage2_df: pd.DataFrame):
    needed = {"farm", "is_fault", "detected", "detected_window"}
    if stage2_df.empty or not needed.issubset(stage2_df.columns):
        return

    rows = []
    for farm, g in stage2_df.groupby("farm"):
        fault = g[g["is_fault"] == 1]
        if fault.empty:
            continue

        cls_acc = np.nan
        if {"fault_type", "pred_type"}.issubset(fault.columns):
            cls_acc = (fault["fault_type"].astype(str) == fault["pred_type"].astype(str)).mean()

        rows.append({
            "farm": farm,
            "det_5consec": fault["detected"].mean(),
            "det_window": fault["detected_window"].mean(),
            "cls_acc": cls_acc,
        })

    plot_df = pd.DataFrame(rows).sort_values("farm")
    if plot_df.empty:
        return

    x = np.arange(len(plot_df))
    width = 0.25

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width, plot_df["det_5consec"], width=width, label="Detection (5 consecutive)")
    ax.bar(x, plot_df["det_window"], width=width, label="Detection (window)")
    ax.bar(x + width, plot_df["cls_acc"].fillna(0), width=width, label="Classification accuracy")

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["farm"])
    ax.set_title("Stage 2 performance by farm")
    ax.set_xlabel("Farm")
    ax.set_ylabel("Rate")
    ax.set_ylim(0, 1.05)
    ax.legend()

    save_fig(fig, "03_stage2_performance_by_farm.png")


def plot_cross_farm_transfer(stage3_df: pd.DataFrame):
    if stage3_df.empty:
        return

    direction_col = None
    comparable_col = None
    for c in stage3_df.columns:
        cl = str(c).lower().strip()
        if cl == "direction":
            direction_col = c
        elif cl == "comparable_rate":
            comparable_col = c

    if direction_col is None or comparable_col is None:
        return

    plot_df = stage3_df[[direction_col, comparable_col]].copy()
    plot_df.columns = ["direction", "comparable_rate"]
    plot_df["comparable_rate"] = pd.to_numeric(plot_df["comparable_rate"], errors="coerce")
    plot_df = plot_df.dropna(subset=["comparable_rate"])
    if plot_df.empty:
        return

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(plot_df["direction"], plot_df["comparable_rate"])
    ax.set_title("Cross-farm transfer rates")
    ax.set_xlabel("Transfer direction")
    ax.set_ylabel("Comparable-fault detection rate")
    ax.set_ylim(0, max(0.25, plot_df["comparable_rate"].max() + 0.05))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    save_fig(fig, "04_cross_farm_transfer_rates.png")


def plot_stage2_selected_cols(stage2_df: pd.DataFrame):
    if stage2_df.empty or not {"farm", "n_selected_cols"}.issubset(stage2_df.columns):
        return

    plot_df = stage2_df.groupby("farm", as_index=False)["n_selected_cols"].mean().sort_values("farm")
    if plot_df.empty:
        return

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(plot_df["farm"], plot_df["n_selected_cols"])
    ax.set_title("Average Stage 2 selected columns by farm")
    ax.set_xlabel("Farm")
    ax.set_ylabel("Average selected columns")

    for i, v in enumerate(plot_df["n_selected_cols"]):
        ax.text(i, v + 1, f"{v:.1f}", ha="center", va="bottom")

    save_fig(fig, "05_stage2_selected_columns_by_farm.png")


def plot_sensor_architecture_comparison():
    groups = [
        "gearbox_bearing", "gearbox_oil", "hydraulic", "transformer",
        "electrical", "generator_bearing", "generator_stator",
        "pitch", "vibration", "rotor_bearing", "wind", "environment",
    ]
    labels = [g.replace("_", " ").title() for g in groups]

    counts = {
        "A": [1, 0, 1, 3, 14, 2, 3, 0, 0, 0, 1, 2],
        "B": [4, 2, 0, 9, 16, 2, 0, 6, 3, 2, 4, 2],
        "C": [16, 16, 15, 1, 53, 0, 7, 24, 4, 8, 3, 13],
    }

    x = np.arange(len(groups))
    width = 0.25
    fig, ax = plt.subplots(figsize=(14, 6))

    for i, farm in enumerate(["A", "B", "C"]):
        ax.bar(
            x + (i - 1) * width,
            counts[farm],
            width,
            label=f"Farm {farm} ({sum(counts[farm])} sensors shown)",
            alpha=0.82,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=9)
    ax.set_ylabel("Number of sensors in group", fontsize=11)
    ax.set_title(
        "Sensor Architecture Comparison Across Farms\n"
        "Farm A (onshore Portugal) vs Farm B & C (offshore Germany)\n"
        "Sparse onshore vs richly instrumented offshore",
        fontsize=11, fontweight="bold"
    )
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3)

    save_fig(fig, "C_sensor_architecture_comparison.png")


def plot_failure_mode_framework():
    fig, ax = plt.subplots(figsize=(13, 7))
    ax.axis("off")

    rows = [
        ["Gearbox", "Bearing wear / oil contamination", "Bearing temp, oil temp, oil pressure, vibration", "High if gearbox sensors exist"],
        ["Generator", "Bearing / stator fault", "Bearing temp, winding temp, phase current/voltage", "Medium–High"],
        ["Transformer", "Insulation overtemperature", "Transformer temp, winding temp, current/voltage", "High when transformer coverage exists"],
        ["Hydraulic", "Pump/coupling/pressure fault", "Pressure, oil temp, brake/azimuth hydraulic channels", "High only with hydraulic sensors"],
        ["Pitch", "Electrical or mechanical fault", "Pitch motor, DC-link, battery/capacitor, angle/position", "High in Farm C"],
        ["Converter", "IGBT / torque-related fault", "DC-link, voltage, current, converter channels", "Low unless converter instrumentation exists"],
    ]

    tbl = ax.table(
        cellText=rows,
        colLabels=["Subsystem", "Failure mode", "Typical SCADA indicators", "Detectability"],
        cellLoc="left",
        loc="center",
        colWidths=[0.12, 0.22, 0.42, 0.24],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 2.5)

    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_text_props(fontweight="bold")
            cell.set_facecolor("#dddddd")
        cell.set_edgecolor("#bbbbbb")

    ax.set_title("Failure Mode Framework and Expected SCADA Indicators", fontsize=12, fontweight="bold")
    save_fig(fig, "N_failure_mode_framework.png")


# ============================================================================
# EVENT BEHAVIOUR PLOTS
# ============================================================================

def plot_event_sensor_behaviour(
    farm: str,
    event_id: str,
    kind: str,
    fault_type: str,
    raw_path: Path,
    fixed_sensors: List[str]
) -> bool:
    df = safe_read_csv(raw_path)
    if df is None or df.empty:
        return False

    meta_map = build_sensor_metadata_map(farm)
    _, pred_df = split_train_pred(df)
    plot_df = pred_df if not pred_df.empty else df

    sensors = [s for s in fixed_sensors if s in plot_df.columns]
    if not sensors:
        return False

    fig, ax = plt.subplots(figsize=(11, 6))
    for s in sensors:
        pretty = pretty_sensor_name(s, farm, meta_map)[:80]
        ax.plot(normalise_series(plot_df[s]).reset_index(drop=True), label=pretty)

    title_fault = fault_type if kind == "fault" else "healthy"
    ax.set_title(f"Farm {farm} {event_id}: {title_fault} {'fault/anomaly' if kind == 'fault' else 'state'} sensor behaviour")
    ax.set_xlabel("Prediction-window timestep")
    ax.set_ylabel("Normalised sensor value")
    ax.legend(fontsize=8)

    save_fig(fig, f"10_event_behaviour_farm{farm}_{kind}_{event_id}.png")
    return True


def plot_event_degradation_proxy(
    farm: str,
    event_id: str,
    kind: str,
    fault_type: str,
    raw_path: Path,
    fixed_sensors: List[str]
) -> bool:
    df = safe_read_csv(raw_path)
    if df is None or df.empty:
        return False

    _, pred_df = split_train_pred(df)
    plot_df = pred_df if not pred_df.empty else df

    sensors = [s for s in fixed_sensors if s in plot_df.columns]
    if not sensors:
        return False

    num_df = pd.DataFrame({s: normalise_series(plot_df[s]) for s in sensors})
    degradation = num_df.abs().mean(axis=1)
    degradation_smooth = moving_mean(degradation, w=6)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(degradation.reset_index(drop=True), label="Instantaneous deviation")
    ax.plot(degradation_smooth.reset_index(drop=True), label="Smoothed deviation", linewidth=2)
    ax.set_title(f"Farm {farm} {event_id}: degradation / anomaly proxy")
    ax.set_xlabel("Prediction-window timestep")
    ax.set_ylabel("Mean absolute z-score")
    ax.legend()

    save_fig(fig, f"11_degradation_proxy_farm{farm}_{kind}_{event_id}.png")
    return True


def plot_train_vs_pred_comparison(
    farm: str,
    event_id: str,
    fault_type: str,
    raw_path: Path,
    fixed_sensors: List[str]
) -> bool:
    df = safe_read_csv(raw_path)
    if df is None or df.empty:
        return False

    meta_map = build_sensor_metadata_map(farm)
    train_df, pred_df = split_train_pred(df)
    if train_df.empty or pred_df.empty:
        return False

    sensors = [s for s in fixed_sensors if s in train_df.columns and s in pred_df.columns]
    if not sensors:
        return False

    rows = []
    for s in sensors:
        rows.append({
            "sensor": s,
            "train_mean": pd.to_numeric(train_df[s], errors="coerce").mean(),
            "pred_mean": pd.to_numeric(pred_df[s], errors="coerce").mean(),
        })

    plot_df = pd.DataFrame(rows)
    if plot_df.empty:
        return False

    x = np.arange(len(plot_df))
    width = 0.35

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width / 2, plot_df["train_mean"], width=width, label="Healthy/train mean")
    ax.bar(x + width / 2, plot_df["pred_mean"], width=width, label="Prediction-window mean")

    pretty_labels = [pretty_sensor_name(s, farm, meta_map)[:55] for s in plot_df["sensor"]]
    ax.set_xticks(x)
    ax.set_xticklabels(pretty_labels, rotation=25, ha="right")
    ax.set_title(f"Farm {farm} {event_id}: healthy vs prediction-window sensor means")
    ax.set_xlabel("Sensor")
    ax.set_ylabel("Mean value")
    ax.legend()

    save_fig(fig, f"12_train_vs_pred_farm{farm}_{event_id}.png")
    return True


def plot_top_sensor_shift(
    farm: str,
    event_id: str,
    fault_type: str,
    raw_path: Path,
    fixed_sensors: List[str]
) -> bool:
    df = safe_read_csv(raw_path)
    if df is None or df.empty:
        return False

    meta_map = build_sensor_metadata_map(farm)
    train_df, pred_df = split_train_pred(df)
    if train_df.empty or pred_df.empty:
        return False

    train_num = coerce_numeric_sensor_block(train_df)
    pred_num = coerce_numeric_sensor_block(pred_df)

    sensors = [s for s in fixed_sensors if s in train_num.columns and s in pred_num.columns]
    if not sensors:
        return False

    rows = []
    for s in sensors:
        shift = score_sensor_shift(train_num[s], pred_num[s])
        rows.append({
            "sensor": s,
            "pretty": pretty_sensor_name(s, farm, meta_map),
            "shift": shift,
        })

    plot_df = pd.DataFrame(rows).sort_values("shift", ascending=False)
    if plot_df.empty:
        return False

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(plot_df["pretty"].str.slice(0, 60), plot_df["shift"])
    ax.set_title(f"Farm {farm} {event_id}: top shifted sensors from healthy baseline")
    ax.set_xlabel("Sensor")
    ax.set_ylabel("Absolute mean shift (z units)")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    save_fig(fig, f"13_top_sensor_shift_farm{farm}_{event_id}.png")
    return True


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("=" * 76)
    print("SCRIPT 5 — DISSERTATION FIGURES (FAULT-TYPE-AWARE, SHARED-SENSOR VERSION)")
    print("=" * 76)
    print(f"Base directory: {BASE_DIR}")
    print()

    ensure_dirs()

    print("Loading outputs...")
    stage1_df = normalise_stage1_event_df(load_stage1_event_df())
    stage1_summary = load_stage1_summary()
    stage2_df = normalise_stage2_df(load_stage2_df())
    stage3_df = load_stage3_summary()

    if STAGE1_JSON.exists():
        print(f"  Stage 1 JSON: {STAGE1_JSON}")
    if STAGE3_JSON.exists():
        print(f"  Stage 3 JSON: {STAGE3_JSON}")
    elif STAGE3_ALT_JSON.exists():
        print(f"  Stage 3 JSON: {STAGE3_ALT_JSON}")

    print(f"  Stage 1 events : {len(stage1_df)}")
    print(f"  Stage 2 events : {len(stage2_df)}")
    print(f"  Stage 3 summary: {len(stage3_df)}")
    print()

    print("Generating performance figures...")
    plot_care_score_by_farm(stage1_summary)
    plot_care_components(stage1_summary)
    plot_stage2_performance(stage2_df)
    plot_cross_farm_transfer(stage3_df)
    plot_stage2_selected_cols(stage2_df)
    plot_sensor_architecture_comparison()
    plot_failure_mode_framework()
    print()

    print("Indexing event CSV files...")
    raw_index = index_raw_event_files()
    print()

    print("Generating event behaviour figures...")
    pairs = choose_example_event_pairs(stage1_df, stage2_df)
    manifest_rows = []

    for farm, pair in pairs.items():
        healthy_event = pair["healthy_event"]
        fault_event = pair["fault_event"]
        fault_type = pair["fault_type"]

        shared_sensors = choose_shared_pair_sensors(
            farm=farm,
            fault_event_id=fault_event,
            fault_type=fault_type,
            raw_index=raw_index,
            max_cols=4
        )

        if not shared_sensors:
            print(f"  Could not determine shared sensors for Farm {farm} pair ({healthy_event}, {fault_event})")
            continue

        healthy_path = raw_index.get((farm, healthy_event))
        if healthy_path is None:
            print(f"  Missing raw CSV for Farm {farm} {healthy_event}")
        else:
            ok1 = plot_event_sensor_behaviour(farm, healthy_event, "healthy", fault_type, healthy_path, shared_sensors)
            ok2 = plot_event_degradation_proxy(farm, healthy_event, "healthy", fault_type, healthy_path, shared_sensors)
            ok3 = plot_train_vs_pred_comparison(farm, healthy_event, fault_type, healthy_path, shared_sensors)
            ok4 = plot_top_sensor_shift(farm, healthy_event, fault_type, healthy_path, shared_sensors)

            if any([ok1, ok2, ok3, ok4]):
                manifest_rows.append({
                    "farm": farm,
                    "event_id": healthy_event,
                    "kind": "healthy",
                    "fault_type_reference": fault_type,
                    "shared_sensors": shared_sensors,
                    "raw_path": str(healthy_path),
                })

        fault_path = raw_index.get((farm, fault_event))
        if fault_path is None:
            print(f"  Missing raw CSV for Farm {farm} {fault_event}")
        else:
            ok1 = plot_event_sensor_behaviour(farm, fault_event, "fault", fault_type, fault_path, shared_sensors)
            ok2 = plot_event_degradation_proxy(farm, fault_event, "fault", fault_type, fault_path, shared_sensors)
            ok3 = plot_train_vs_pred_comparison(farm, fault_event, fault_type, fault_path, shared_sensors)
            ok4 = plot_top_sensor_shift(farm, fault_event, fault_type, fault_path, shared_sensors)

            if any([ok1, ok2, ok3, ok4]):
                manifest_rows.append({
                    "farm": farm,
                    "event_id": fault_event,
                    "kind": "fault",
                    "fault_type": fault_type,
                    "shared_sensors": shared_sensors,
                    "raw_path": str(fault_path),
                })

    manifest = {
        "base_dir": str(BASE_DIR),
        "stage1_csv": str(STAGE1_EVENT_CSV) if STAGE1_EVENT_CSV.exists() else None,
        "stage1_summary_csv": str(STAGE1_SUMMARY_CSV) if STAGE1_SUMMARY_CSV.exists() else None,
        "stage2_csv": str(STAGE2_CSV) if STAGE2_CSV.exists() else None,
        "stage3_summary_csv": str(STAGE3_SUMMARY_CSV) if STAGE3_SUMMARY_CSV.exists() else None,
        "stage3_json": str(STAGE3_JSON) if STAGE3_JSON.exists() else (str(STAGE3_ALT_JSON) if STAGE3_ALT_JSON.exists() else None),
        "indexed_raw_events": len(raw_index),
        "paired_example_events_plotted": manifest_rows,
    }

    manifest_path = OUTPUT_DIR / "figure_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(json_safe(manifest), f, indent=2)

    print()
    print("Saving reports...")
    print(f"  Saved: {manifest_path}")
    print()
    print("=" * 76)
    print("SCRIPT 5 COMPLETE")
    print("=" * 76)
    print(f"Plots -> {PLOTS_DIR}")
    print(f"Reports -> {REPORTS_DIR}")


if __name__ == "__main__":
    main()

SCRIPT 5 — DISSERTATION FIGURES (FAULT-TYPE-AWARE, SHARED-SENSOR VERSION)
Base directory: E:\project\CARE_To_Compare

Loading outputs...
  Stage 1 CSV : E:\project\CARE_To_Compare\pipeline_final_v5\event_results_final.csv
  Stage 2 CSV : E:\project\CARE_To_Compare\fault_classification_outputs\fault_classification_results.csv
  Stage 1 JSON: E:\project\CARE_To_Compare\pipeline_final_v5\results_final.json
  Stage 3 JSON: E:\project\CARE_To_Compare\cross_farm_outputs\cross_farm_results.json
  Stage 1 events : 95
  Stage 2 events : 95
  Stage 3 summary: 5

Generating performance figures...
  Saved: E:\project\CARE_To_Compare\dissertation_figures_outputs\plots\01_care_score_by_farm.png
  Saved: E:\project\CARE_To_Compare\dissertation_figures_outputs\plots\02_care_component_breakdown.png
  Saved: E:\project\CARE_To_Compare\dissertation_figures_outputs\plots\03_stage2_performance_by_farm.png
  Saved: E:\project\CARE_To_Compare\dissertation_figures_outputs\plots\05_stage2_selected_columns_by_f